> **Public snapshot note**
> 이 노트북은 원래 연구 실행 흐름을 보존한 archival artifact입니다.
> 공개 스냅샷에서는 raw PDF, processed corpus, `kg_gen/graphrag_workspace/input`, `kg_gen/graphrag_workspace/output`,
> `kg_gen/snu_kg_output/graphrag_workspace/output*` 등 bulk/source-derived 자산을 의도적으로 제외했습니다.
> 따라서 아래의 일부 경로 설명은 **원래 실험 환경 기준**이며, 현재 public snapshot과 1:1로 대응하지 않을 수 있습니다.


## Phase 5 재설계: GraphRAG 기반 정답 구조 계산

### 새로운 접근 방식
1. **그래프에서 정답 구조(answer_struct) 계산**
   - Multi-hop: NetworkX로 실제 경로 찾기
   - Community: 실제 중심성/통계 계산
   - Cross-context: 실제 text unit 분석
   - Implicit: 실제 공통 이웃 찾기
   - Causal: 실제 인과 경로 추적

2. **LLM은 표현만 담당**
   - 입력: answer_struct + evidence_texts
   - 역할: 구조를 자연어로 변환
   - 금지: 새로운 정보 추가

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd
import networkx as nx
import numpy as np
from pathlib import Path
import json
import pickle
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Set, Optional, Any
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 임포트 완료")

In [ ]:
# 경로 설정 및 데이터 로드
base_path = Path("<PROJECT_ROOT>")
kg_path = base_path / "kg_gen/snu_kg_output/graphrag_workspace/output_after_kggen"
output_path = base_path / "rag_dataset/kg_qa_datasets"

print("=== Phase 2 그래프 데이터 로드 ===")
# NetworkX 그래프 로드
with open(output_path / 'networkx_graph_data.pkl', 'rb') as f:
    graph_data = pickle.load(f)

G = graph_data['graph']
nodes_by_community = graph_data['nodes_by_community']
nodes_by_text_unit = graph_data['nodes_by_text_unit']
edges_by_text_unit = graph_data['edges_by_text_unit']
centrality_measures = graph_data['centrality_measures']

print(f"✓ NetworkX 그래프 로드: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Phase 1 텍스트 데이터 로드
with open(output_path / 'processed_graphrag_data.pkl', 'rb') as f:
    processed_data = pickle.load(f)

text_unit_lookup = processed_data['text_unit_lookup']
entity_to_communities = processed_data['entity_to_communities']
community_reports_lookup = processed_data['community_reports_lookup']
entities_by_title = processed_data['entities_by_title']

print("✓ 텍스트 및 메타데이터 로드 완료")

# Phase 4에서 생성된 질문들 로드
print("\n=== Phase 4 질문 데이터 로드 ===")
all_questions = {}
qa_types = ['multi_hop', 'community_synthesis', 'cross_context', 'implicit_relationships', 'causal_chains']

for qa_type in qa_types:
    file_path = output_path / qa_type / 'generated_questions.json'
    with open(file_path, 'r', encoding='utf-8') as f:
        questions = json.load(f)
        all_questions[qa_type] = questions
        print(f"✓ {qa_type}: {len(questions)}개 질문 로드")

total_questions = sum(len(q) for q in all_questions.values())
print(f"\n총 {total_questions}개 질문 로드 완료")

## Cell 36: Phase 5 설정 - GPT-5.2 기반 GT 생성 시스템

In [ ]:
# Phase 5 설정 - GPT-4 기반 GT 생성 시스템 구현
import openai
from openai import OpenAI
import time
from tqdm import tqdm
import logging
from typing import Dict, List, Optional, Tuple
import json
import os
from datetime import datetime

# OpenAI API 키 설정
import getpass

if "OPENAI_API_KEY" not in os.environ:
    print("⚠️ OpenAI API 키가 환경 변수에 설정되지 않았습니다.")
    print("\nAPI 키를 입력하세요 (입력 내용은 화면에 표시되지 않습니다):")
    api_key = getpass.getpass("OpenAI API Key: ")
    
    if api_key:
        os.environ['OPENAI_API_KEY'] = api_key
        print("✅ API 키가 설정되었습니다.")
    else:
        print("❌ API 키를 입력하지 않았습니다.")
        print("\n다음 중 하나의 방법으로 설정할 수 있습니다:")
        print("1. 이 셀을 다시 실행하여 API 키 입력")
        print("2. 터미널에서: export OPENAI_API_KEY='your-api-key'")
else:
    print("✅ API 키가 이미 설정되어 있습니다.")
    print("\n다시 설정하려면:")
    response = input("새 API 키를 입력하시겠습니까? (y/n): ")
    if response.lower() == 'y':
        api_key = getpass.getpass("새 OpenAI API Key: ")
        if api_key:
            os.environ['OPENAI_API_KEY'] = api_key
            print("✅ API 키가 업데이트되었습니다.")

# API 키 설정 확인 (처음 몇 자리만 표시)
if "OPENAI_API_KEY" in os.environ and os.environ['OPENAI_API_KEY']:
    key_preview = os.environ['OPENAI_API_KEY'][:8] + "..." if len(os.environ['OPENAI_API_KEY']) > 8 else "설정됨"
    print(f"\n현재 API 키: {key_preview}")

# OpenAI 클라이언트 초기화
client = OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY')
)

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('gt_generation.log'),
        logging.StreamHandler()
    ]
)

# GPT-4 모델 설정 (GPT-5.2가 아직 없으므로 GPT-4 사용)
MODEL_NAME = "gpt-4o-mini"  # 또는 "gpt-4-turbo-preview"
MAX_RETRIES = 3
RETRY_DELAY = 2
BATCH_SIZE = 10

print(f"GT 생성 시스템 초기화 완료")
print(f"사용 모델: {MODEL_NAME}")
print(f"API 키 설정: {'✓' if os.getenv('OPENAI_API_KEY') else '✗'}")

# 재시도 데코레이터
def retry_with_exponential_backoff(func):
    """지수 백오프를 사용한 재시도 데코레이터"""
    def wrapper(*args, **kwargs):
        for attempt in range(MAX_RETRIES):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    raise e
                wait_time = RETRY_DELAY * (2 ** attempt)
                logging.warning(f"시도 {attempt + 1} 실패: {str(e)}. {wait_time}초 대기...")
                time.sleep(wait_time)
        return None
    return wrapper

## Cell 37: Multi-hop 정답 구조 계산

Multi-hop 질문에 대해 그래프에서 실제 경로를 찾고 관련 정보를 추출합니다.

In [ ]:
# 개선된 Multi-hop 정답 구조 계산 함수들
from typing import Any, Dict, List, Optional, Tuple, Iterable
import networkx as nx
import re

def _iter_out_neighbors(G: nx.Graph, u: Any) -> Iterable[Any]:
    """Directed면 successors, 아니면 neighbors."""
    if G.is_directed():
        return G.successors(u)
    return G.neighbors(u)

def _edge_attr_dicts(G: nx.Graph, u: Any, v: Any) -> List[Dict[str, Any]]:
    """
    (u,v) 사이 엣지 속성 dict들을 리스트로 반환.
    - MultiGraph/MultiDiGraph: key별 dict 여러 개
    - Graph/DiGraph: 단일 dict
    """
    data = G.get_edge_data(u, v, default=None)
    if data is None:
        return []
    if G.is_multigraph():
        # data: {key: attr_dict, ...}
        return [attr for _, attr in data.items()]
    # data: attr_dict
    return [data]

def _get_relation_type(edge_attr: Dict[str, Any]) -> str:
    return (
        edge_attr.get("relation")
        or edge_attr.get("type")
        or edge_attr.get("label")
        or ""
    )

def _has_evidence(edge_attr: Dict[str, Any]) -> bool:
    # 가능한 증거 필드들
    if "text_unit_ids" in edge_attr and edge_attr["text_unit_ids"]:
        return True
    if "text_unit_id" in edge_attr and edge_attr["text_unit_id"]:
        return True
    if "original_texts" in edge_attr and edge_attr["original_texts"]:
        return True
    return False

def find_k_hop_unique_path(
    G: nx.Graph,
    start: Any,
    end: Any,
    k: int,
    allowed_rel_types: Optional[set] = None,
    require_evidence_on_edges: bool = True,
    max_paths_to_check: int = 50000,
) -> Dict[str, Any]:
    """
    정확히 k-hop(=엣지 k개)인 경로가 '유일'한지 판단하고, 유일하면 그 경로를 반환.
    
    Returns dict:
      {
        "exists": bool,
        "unique": bool,
        "path": [n0, n1, ..., nk] or [],
        "num_paths_found": int,
        "ambiguous": bool,
        "reason": str
      }
    """
    if k <= 0:
        return {"exists": False, "unique": False, "path": [], "num_paths_found": 0, "ambiguous": False,
                "reason": "k must be >= 1"}

    if start not in G or end not in G:
        return {"exists": False, "unique": False, "path": [], "num_paths_found": 0, "ambiguous": False,
                "reason": "start or end not in graph"}

    # 빠른 shortcut 체크(멀티홉 QA 보호)
    if k >= 2:
        if G.has_edge(start, end) or (not G.is_directed() and G.has_edge(end, start)):
            return {
                "exists": True,
                "unique": False,
                "path": [],
                "num_paths_found": 0,
                "ambiguous": True,
                "reason": "direct edge exists (shortcut), reject for multi-hop GT"
            }

    # DFS로 정확히 k hop 경로를 카운트
    found_path: Optional[List[Any]] = None
    count = 0

    def dfs(curr: Any, depth: int, path: List[Any], visited: set):
        nonlocal found_path, count
        if count >= max_paths_to_check:
            return

        if depth == k:
            if curr == end:
                count += 1
                if count == 1:
                    found_path = path.copy()
            return

        for nxt in _iter_out_neighbors(G, curr):
            if nxt in visited:
                continue

            # (curr -> nxt) 엣지 중 하나라도 조건을 만족하면 진행
            attr_list = _edge_attr_dicts(G, curr, nxt)
            if not attr_list:
                continue

            ok = False
            for attr in attr_list:
                rel = _get_relation_type(attr)
                if allowed_rel_types is not None and rel and rel not in allowed_rel_types:
                    continue
                if require_evidence_on_edges and not _has_evidence(attr):
                    continue
                ok = True
                break

            if not ok:
                continue

            visited.add(nxt)
            path.append(nxt)
            dfs(nxt, depth + 1, path, visited)
            path.pop()
            visited.remove(nxt)

            # 유일성만 필요하므로 2개 이상 찾으면 조기 종료
            if count >= 2:
                return

    dfs(start, 0, [start], {start})

    if count == 0:
        return {"exists": False, "unique": False, "path": [], "num_paths_found": 0, "ambiguous": False,
                "reason": "no k-hop path found"}
    if count == 1 and found_path is not None:
        return {"exists": True, "unique": True, "path": found_path, "num_paths_found": 1, "ambiguous": False,
                "reason": "unique k-hop path found"}
    return {"exists": True, "unique": False, "path": [], "num_paths_found": count, "ambiguous": True,
            "reason": "multiple k-hop paths found (ambiguous)"}

def collect_evidence_text_unit_ids(edge_attr: Dict[str, Any], top_k: int = 3) -> List[str]:
    """
    엣지 속성에서 text_unit_ids를 최대 top_k개 추출.
    """
    ids: List[str] = []

    if "text_unit_ids" in edge_attr and edge_attr["text_unit_ids"]:
        raw = edge_attr["text_unit_ids"]
        if isinstance(raw, list):
            ids.extend([str(x) for x in raw])
        else:
            ids.append(str(raw))

    if not ids and "text_unit_id" in edge_attr and edge_attr["text_unit_id"]:
        ids.append(str(edge_attr["text_unit_id"]))

    # original_texts 내부의 id 추출
    if not ids and "original_texts" in edge_attr and edge_attr["original_texts"]:
        raw = edge_attr["original_texts"]
        if isinstance(raw, list):
            for item in raw:
                if isinstance(item, dict):
                    for key in ("text_unit_id", "tu_id", "id"):
                        if key in item and item[key]:
                            ids.append(str(item[key]))
                            break

    # 중복 제거(순서 유지)
    seen = set()
    dedup = []
    for x in ids:
        if x not in seen:
            seen.add(x)
            dedup.append(x)

    return dedup[:top_k]

def _pick_best_edge_attr(
    edge_attrs: List[Dict[str, Any]],
    allowed_rel_types: Optional[set] = None,
    require_evidence: bool = True,
) -> Optional[Dict[str, Any]]:
    """
    (u,v) 사이 후보 엣지들 중 대표 엣지를 결정적으로 선택.
    """
    candidates = []
    for attr in edge_attrs:
        rel = _get_relation_type(attr)
        if allowed_rel_types is not None and rel and rel not in allowed_rel_types:
            continue
        if require_evidence and not _has_evidence(attr):
            continue
        ev = collect_evidence_text_unit_ids(attr, top_k=10)
        weight = attr.get("weight", 0) or 0
        desc = attr.get("description", "") or ""
        candidates.append((len(ev), float(weight), len(desc), rel, desc, attr))

    if not candidates:
        return None

    # 정렬: evidence 수, weight, description 길이, rel/desc 사전순
    candidates.sort(key=lambda x: (x[0], x[1], x[2], x[3], x[4]), reverse=True)
    return candidates[0][-1]

def build_answer_struct_from_path(
    G: nx.Graph,
    path: List[Any],
    allowed_rel_types: Optional[set] = None,
    require_evidence_on_edges: bool = True,
    evidence_top_k_per_edge: int = 3,
    include_node_evidence: bool = False,
    node_evidence_top_k: int = 1,
) -> Dict[str, Any]:
    """
    path(노드 리스트)로부터 answer_struct를 구성.
    """
    if not path or len(path) < 2:
        return {
            "path": [],
            "path_exists": False,
            "hop_count": 0,
            "edges": [],
            "evidence_text_unit_ids_by_edge": [],
            "evidence_text_unit_ids": []  # 추가!
        }

    edges_out = []
    evidence_out = []
    all_evidence_ids = []  # 모든 evidence 수집

    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        attr_list = _edge_attr_dicts(G, u, v)
        chosen = _pick_best_edge_attr(
            attr_list,
            allowed_rel_types=allowed_rel_types,
            require_evidence=require_evidence_on_edges,
        )
        if chosen is None:
            return {
                "path": path,
                "path_exists": False,
                "hop_count": len(path) - 1,
                "edges": [],
                "evidence_text_unit_ids_by_edge": [],
                "evidence_text_unit_ids": [],  # 추가!
                "reason": f"no valid edge for hop {u} -> {v}"
            }

        rel = _get_relation_type(chosen) or "related_to"
        edge_obj = {"source": u, "relation": rel, "target": v}

        tu_ids = collect_evidence_text_unit_ids(chosen, top_k=evidence_top_k_per_edge)
        all_evidence_ids.extend(tu_ids)  # 전체 evidence에 추가

        edges_out.append(edge_obj)
        evidence_out.append({
            "edge": edge_obj,
            "text_unit_ids": tu_ids
        })

    answer = {
        "start_node": path[0],
        "end_node": path[-1],
        "path": path,
        "path_exists": True,
        "hop_count": len(path) - 1,
        "edges": edges_out,
        "evidence_text_unit_ids_by_edge": evidence_out,
        "evidence_text_unit_ids": all_evidence_ids  # 추가!
    }

    if include_node_evidence:
        node_ev = []
        for n in path:
            nd = G.nodes.get(n, {})
            ids = []
            if "text_unit_ids" in nd and nd["text_unit_ids"]:
                raw = nd["text_unit_ids"]
                if isinstance(raw, list):
                    ids = [str(x) for x in raw][:node_evidence_top_k]
                else:
                    ids = [str(raw)]
                all_evidence_ids.extend(ids)  # 노드 evidence도 추가
            node_ev.append({"node": n, "text_unit_ids": ids})
        answer["evidence_text_unit_ids_by_node"] = node_ev

    return answer

def extract_nodes_from_multihop_question(question: str) -> Optional[Dict[str, str]]:
    """
    Multi-hop 질문에서 노드들을 추출
    패턴: "A에서 B을(를) 거쳐 C로 이어지는 관계를 설명하세요."
    """
    pattern = r"(.+?)에서\s+(.+?)을\(를\)\s*거쳐\s+(.+?)로"
    match = re.search(pattern, question)
    
    if match:
        return {
            'start_node': match.group(1).strip(),
            'middle_node': match.group(2).strip(),
            'end_node': match.group(3).strip(),
            'hop_count': 2  # 현재는 2-hop만 지원
        }
    return None

def compute_multihop_answer_struct(question: str, pattern_data: Dict[str, Any] = None) -> Dict[str, Any]:
    """
    Multi-hop 질문에 대한 정답 구조 계산 (개선된 버전)
    
    pattern_data가 없으면 질문에서 직접 추출
    """
    # Pattern data가 없으면 질문에서 추출
    if not pattern_data or 'start_node' not in pattern_data:
        extracted = extract_nodes_from_multihop_question(question)
        if not extracted:
            return {
                "path_exists": False,
                "unique": False,
                "reason": "cannot extract nodes from question",
                "path": [],
                "hop_count": 0,
                "edges": [],
                "evidence_text_unit_ids_by_edge": [],
                "evidence_text_unit_ids": []
            }
        pattern_data = extracted
    
    start_node = pattern_data.get('start_node')
    end_node = pattern_data.get('end_node') 
    hop_count = pattern_data.get('hop_count', 2)
    
    if not start_node or not end_node:
        return {
            "path_exists": False,
            "unique": False,
            "reason": "start or end node not in pattern",
            "path": [],
            "hop_count": 0,
            "edges": [],
            "evidence_text_unit_ids_by_edge": [],
            "evidence_text_unit_ids": []
        }
    
    # 정확히 k-hop 경로 찾기
    res = find_k_hop_unique_path(
        G,
        start=start_node,
        end=end_node,
        k=hop_count,
        allowed_rel_types=None,  # 필요시 관계 타입 제한 추가
        require_evidence_on_edges=True,
    )
    
    if not (res["exists"] and res["unique"]):
        return {
            "path_exists": res["exists"],
            "unique": res.get("unique", False),
            "ambiguous": res.get("ambiguous", False),
            "reason": res.get("reason", ""),
            "start_node": start_node,
            "end_node": end_node,
            "path": [],
            "hop_count": 0,
            "edges": [],
            "evidence_text_unit_ids_by_edge": [],
            "evidence_text_unit_ids": []
        }

    # answer_struct 구성
    answer = build_answer_struct_from_path(
        G,
        path=res["path"],
        allowed_rel_types=None,
        require_evidence_on_edges=True,
        evidence_top_k_per_edge=3,
        include_node_evidence=False,
    )
    answer["unique"] = True
    answer["reason"] = res.get("reason", "")
    return answer

# 테스트
print("=== 개선된 Multi-hop 정답 구조 계산 테스트 ===")
# 첫 번째 multi-hop 질문으로 테스트
if all_questions.get('multi_hop'):
    test_q = all_questions['multi_hop'][0]
    print(f"질문: {test_q['question']}")
    print(f"패턴: {test_q.get('pattern_id', 'N/A')}")
    
    # pattern_data 없이 질문에서 직접 추출
    answer_struct = compute_multihop_answer_struct(test_q['question'])
    
    print(f"\n계산된 정답 구조:")
    print(f"- 경로 존재: {answer_struct['path_exists']}")
    print(f"- 유일성: {answer_struct.get('unique', False)}")
    print(f"- 이유: {answer_struct.get('reason', '')}")
    print(f"- evidence_text_unit_ids 개수: {len(answer_struct.get('evidence_text_unit_ids', []))}")
    
    if answer_struct['path_exists'] and answer_struct.get('unique'):
        print(f"- 경로: {' → '.join(answer_struct['path'])}")
        print(f"- 홉 수: {answer_struct['hop_count']}")
        print(f"- Evidence IDs: {len(answer_struct['evidence_text_unit_ids_by_edge'])}개 엣지")

## Cell 38: Community Synthesis 정답 구조 계산

Community 관련 질문에 대해 실제 커뮤니티 데이터와 그래프 통계를 기반으로 정답 구조를 계산합니다.

In [ ]:
def compute_community_synthesis_answer_struct(question: str, pattern_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Community synthesis 질문에 대한 정답 구조 계산 (수정된 버전)
    
    Returns:
        answer_struct: {
            'community_id': int,
            'entities': 커뮤니티 엔티티 리스트,
            'entity_count': int,
            'relationships': 커뮤니티 내부 관계 리스트,
            'relationship_count': int,
            'key_entities': 중요 엔티티 (중심성 기반),
            'top_relation_types': 관계 타입 상위 3개,
            'representative_triples': 대표 트리플 3개,
            'density': 커뮤니티 밀도,
            'evidence_by_triple': 트리플별 evidence,
            'evidence_text_unit_ids': 전체 evidence IDs
        }
    """
    # Pattern에서 community_id 추출
    community_id = pattern_data.get('community_id')
    level = pattern_data.get('level', 0)  # 기본값 0
    
    if community_id is None:
        return {
            'community_id': None,
            'entity_count': 0,
            'entities': [],
            'relationship_count': 0,
            'relationships': [],
            'key_entities': [],
            'top_relation_types': [],
            'representative_triples': [],
            'density': 0.0,
            'evidence_by_triple': [],
            'evidence_text_unit_ids': []
        }
    
    # nodes_by_community에서 노드 추출 (중첩 구조 처리)
    community_nodes = []
    
    # level이 nodes_by_community에 있는지 확인
    if level in nodes_by_community:
        level_data = nodes_by_community[level]
        if isinstance(level_data, dict):
            # level_data가 dict인 경우, community_id를 키로 사용
            if community_id in level_data:
                nodes = level_data[community_id]
                if isinstance(nodes, list):
                    community_nodes = nodes
            else:
                # community_id가 없으면 모든 노드를 합쳐서 사용 (임시)
                for sub_comm_id, nodes in level_data.items():
                    if isinstance(nodes, list):
                        community_nodes.extend(nodes)
                        break  # 첫 번째 sub-community만 사용
    
    if not community_nodes:
        return {
            'community_id': community_id,
            'entity_count': 0,
            'entities': [],
            'relationship_count': 0,
            'relationships': [],
            'key_entities': [],
            'top_relation_types': [],
            'representative_triples': [],
            'density': 0.0,
            'evidence_by_triple': [],
            'evidence_text_unit_ids': []
        }
    
    # 커뮤니티 서브그래프 추출
    subgraph = G.subgraph(community_nodes).copy()
    
    # 방향성 통일 (커뮤니티 구조 요약 목적)
    if subgraph.is_directed():
        subgraph_undirected = subgraph.to_undirected()
    else:
        subgraph_undirected = subgraph
    
    evidence_ids = set()
    internal_edges = []
    rel_type_counts = {}
    evidence_by_triple = []
    
    def add_edge(u, v, attr):
        """엣지를 추가하고 관련 정보를 수집"""
        rel_type = _get_relation_type(attr) or "related_to"
        tu_ids = collect_evidence_text_unit_ids(attr, top_k=5)
        
        edge_obj = {
            "source": u, 
            "relation": rel_type, 
            "target": v, 
            "weight": attr.get("weight", 0)
        }
        
        internal_edges.append(edge_obj)
        rel_type_counts[rel_type] = rel_type_counts.get(rel_type, 0) + 1
        
        if tu_ids:
            evidence_ids.update(tu_ids)
            evidence_by_triple.append({
                "triple": {"s": u, "p": rel_type, "o": v},
                "text_unit_ids": tu_ids
            })
    
    # MultiGraph 처리 수정: keys=True로 직접 순회
    if G.is_multigraph():
        for u, v, k, attr in subgraph.edges(keys=True, data=True):
            add_edge(u, v, attr)
    else:
        for u, v, attr in subgraph.edges(data=True):
            add_edge(u, v, attr)
    
    # 중요 엔티티 찾기 (undirected 버전으로 중심성 계산)
    key_entities = []
    if subgraph_undirected.number_of_nodes() > 0:
        degree_cent = nx.degree_centrality(subgraph_undirected)
        sorted_nodes = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)
        
        key_entities = [
            {
                'entity': node,
                'degree_centrality': float(cent),
                'type': G.nodes[node].get('type', 'unknown') if node in G else 'unknown'
            }
            for node, cent in sorted_nodes[:5]
        ]
    
    # 관계 타입 상위 3개 (빈도순)
    top_relation_types = [
        {'relation_type': rt, 'count': count}
        for rt, count in sorted(rel_type_counts.items(), key=lambda x: x[1], reverse=True)[:3]
    ]
    
    # 대표 트리플: evidence가 많은 것 우선
    sorted_triples = sorted(evidence_by_triple, key=lambda x: len(x['text_unit_ids']), reverse=True)[:3]
    representative_triples = [x['triple'] for x in sorted_triples]
    
    # 커뮤니티 밀도 계산
    density = nx.density(subgraph_undirected) if subgraph_undirected.number_of_nodes() > 1 else 0.0
    
    # Key entities의 text unit IDs 수집
    for ke in key_entities:
        node = ke['entity']
        node_data = G.nodes.get(node, {})
        node_ids = node_data.get('text_unit_ids', [])
        
        if isinstance(node_ids, list):
            evidence_ids.update([str(x) for x in node_ids[:2]])
        elif node_ids:
            evidence_ids.add(str(node_ids))
    
    return {
        'community_id': community_id,
        'entities': community_nodes[:20],  # 최대 20개
        'entity_count': len(community_nodes),
        'relationships': internal_edges[:20],  # 최대 20개
        'relationship_count': len(internal_edges),
        'key_entities': key_entities,
        'top_relation_types': top_relation_types,
        'representative_triples': representative_triples,
        'density': round(float(density), 4),
        'evidence_by_triple': sorted_triples,  # triple + text_unit_ids
        'evidence_text_unit_ids': list(evidence_ids)[:15]  # 최대 15개
    }

# 테스트
print("=== Community Synthesis 정답 구조 계산 테스트 (수정 버전) ===")
if all_questions.get('community_synthesis'):
    test_q = all_questions['community_synthesis'][0]
    print(f"질문: {test_q['question'][:100]}...")
    
    # 임시 pattern data - level 0, community_id 1 사용
    pattern_data = {
        'community_id': 1,  # nodes_by_community[0][1]에 해당
        'level': 0
    }
    
    answer_struct = compute_community_synthesis_answer_struct(test_q['question'], pattern_data)
    print(f"\n계산된 정답 구조:")
    print(f"- Community ID: {answer_struct['community_id']}")
    print(f"- 엔티티 수: {answer_struct['entity_count']}")
    print(f"- 관계 수: {answer_struct['relationship_count']}")
    print(f"- 밀도: {answer_struct['density']}")
    print(f"- 주요 엔티티: {len(answer_struct['key_entities'])}개")
    if answer_struct['key_entities']:
        print(f"  - 최중요 엔티티: {answer_struct['key_entities'][0]['entity']} (중심성: {answer_struct['key_entities'][0]['degree_centrality']:.3f})")
    print(f"- 상위 관계 타입: {len(answer_struct['top_relation_types'])}개")
    if answer_struct['top_relation_types']:
        print(f"  - 최다 관계: {answer_struct['top_relation_types'][0]['relation_type']} ({answer_struct['top_relation_types'][0]['count']}회)")
    print(f"- 대표 트리플: {len(answer_struct['representative_triples'])}개")
    print(f"- Evidence by triple: {len(answer_struct['evidence_by_triple'])}개")
    print(f"- 전체 Evidence IDs: {len(answer_struct['evidence_text_unit_ids'])}개")

## Cell 39: Cross-context Integration 정답 구조 계산

여러 문서/텍스트 유닛에 걸친 정보 통합 질문에 대한 정답 구조를 계산합니다.

In [ ]:
# Cross-context Integration을 위한 전역 인덱스 구축
from collections import Counter, defaultdict

def build_text_unit_edge_index(G):
    """
    텍스트 유닛별 트리플 인덱스 구축 (성능 최적화)
    Returns: tu_to_triples = {tu_id: [{"s", "p", "o", "text_unit_ids"}]}
    """
    tu_to_triples = defaultdict(list)
    
    if G.is_multigraph():
        for u, v, k, attr in G.edges(keys=True, data=True):
            rel = _get_relation_type(attr) or "related_to"
            tu_ids = collect_evidence_text_unit_ids(attr, top_k=50)
            
            # 수정 1: 빈 tu_ids는 스킵
            if not tu_ids:
                continue
                
            # description도 저장 (나중에 LLM이 활용)
            desc = attr.get('description', '')
            
            for tu in tu_ids:
                # 수정 7: text_unit_id를 str로 강제
                tu_str = str(tu)
                tu_to_triples[tu_str].append({
                    "s": u, 
                    "p": rel, 
                    "o": v, 
                    "description": desc,
                    "text_unit_ids": [tu_str]
                })
    else:
        for u, v, attr in G.edges(data=True):
            rel = _get_relation_type(attr) or "related_to"
            tu_ids = collect_evidence_text_unit_ids(attr, top_k=50)
            
            # 수정 1: 빈 tu_ids는 스킵
            if not tu_ids:
                continue
                
            desc = attr.get('description', '')
            
            for tu in tu_ids:
                # 수정 7: text_unit_id를 str로 강제
                tu_str = str(tu)
                tu_to_triples[tu_str].append({
                    "s": u, 
                    "p": rel, 
                    "o": v,
                    "description": desc,
                    "text_unit_ids": [tu_str]
                })
    
    return tu_to_triples

# 전역 인덱스 구축 (한 번만 실행)
print("텍스트 유닛 인덱스 구축 중...")
TU_TO_TRIPLES = build_text_unit_edge_index(G)
print(f"완료: {len(TU_TO_TRIPLES)} 텍스트 유닛 인덱싱")

def compute_cross_context_answer_struct(question: str, pattern_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Cross-context Integration 질문에 대한 정답 구조 계산 (최종 개선 버전)
    
    Returns:
        answer_struct: {
            'entity': 중심 엔티티,
            'text_unit_count': 관련 텍스트 유닛 수,
            'text_units': 텍스트 유닛 ID 리스트,
            'info_by_text_unit': 각 텍스트 유닛별 정보,
            'relationships_by_text_unit': 텍스트 유닛별 트리플,
            'cross_references': 텍스트 유닛 간 공통 엔티티,
            'unique_aspects': 각 텍스트 유닛의 고유 정보,
            'unique_triples_by_text_unit': 텍스트 유닛별 고유 트리플,
            'common_themes': 공통 관계 타입,
            'common_triples': 여러 텍스트 유닛에 걸친 트리플,
            'evidence_coverage': 커버리지 통계
        }
    """
    # Pattern에서 entity 추출 (질문 파싱 금지)
    entity_name = pattern_data.get('entity') or pattern_data.get('entity_name')
    
    if not entity_name or entity_name not in G:
        return {
            'entity': entity_name,
            'text_unit_count': 0,
            'text_units': [],
            'info_by_text_unit': [],
            'relationships_by_text_unit': [],
            'cross_references': [],
            'unique_aspects': [],
            'unique_triples_by_text_unit': [],
            'common_themes': [],
            'common_triples': [],
            'evidence_coverage': {}
        }
    
    # 1) entity가 등장하는 text unit들 찾기 (엣지 기반)
    candidate_tus = set()
    
    if G.is_multigraph():
        for u, v, k, attr in G.edges(keys=True, data=True):
            if u == entity_name or v == entity_name:
                tu_ids = collect_evidence_text_unit_ids(attr, top_k=50)
                candidate_tus.update([str(tu) for tu in tu_ids])
    else:
        for u, v, attr in G.edges(data=True):
            if u == entity_name or v == entity_name:
                tu_ids = collect_evidence_text_unit_ids(attr, top_k=50)
                candidate_tus.update([str(tu) for tu in tu_ids])
    
    # 수정 2: anchor triple 수 기준으로 상위 10개 선택
    scored_tus = []
    for tu in candidate_tus:
        triples = TU_TO_TRIPLES.get(tu, [])
        anchor_count = sum(1 for t in triples if t["s"] == entity_name or t["o"] == entity_name)
        if anchor_count > 0:  # anchor triple이 있는 것만
            scored_tus.append((anchor_count, tu))
    
    # anchor triple이 많은 순으로 정렬하여 상위 10개 선택
    scored_tus.sort(reverse=True)
    text_units = [tu for _, tu in scored_tus[:10]]
    
    # 2) 각 텍스트 유닛별 정보 구성 (triple 기반)
    info_by_tu = []
    relationships_by_tu = []
    tu_entities_map = {}
    tu_reltype_map = {}
    tu_triples_map = {}  # 수정 5: 트리플 집합 저장
    
    for tu_id in text_units:
        # 이 텍스트 유닛이 지지하는 모든 트리플
        all_triples = TU_TO_TRIPLES.get(tu_id, [])
        
        # entity가 포함된 트리플만 선택 (더 정밀한 분석)
        anchor_triples = [
            t for t in all_triples 
            if t["s"] == entity_name or t["o"] == entity_name
        ]
        
        # 텍스트 유닛의 엔티티와 관계 타입 (트리플에서 유도)
        entities = set()
        rel_types = set()
        triple_keys = set()  # 수정 5: 트리플 키 저장
        
        for triple in anchor_triples:
            entities.add(triple["s"])
            entities.add(triple["o"])
            rel_types.add(triple["p"])
            triple_keys.add((triple["s"], triple["p"], triple["o"]))
        
        tu_entities_map[tu_id] = entities
        tu_reltype_map[tu_id] = rel_types
        tu_triples_map[tu_id] = triple_keys
        
        info_by_tu.append({
            'text_unit_id': tu_id,
            'entity_count': len(entities),
            'entities': sorted(list(entities))[:10],
            'relation_types': sorted(list(rel_types)),
            'triple_count': len(anchor_triples),
            'total_triple_count': len(all_triples)
        })
        
        # 수정 4: 트리플 정렬하여 결정성 보장
        anchor_triples_sorted = sorted(
            anchor_triples,
            key=lambda t: (t["p"], str(t["s"]), str(t["o"]), t.get("description", ""))
        )
        
        relationships_by_tu.append({
            'text_unit_id': tu_id,
            'triples': anchor_triples_sorted[:5]  # 정렬된 상위 5개
        })
    
    # 3) Cross-references: 여러 텍스트 유닛에 공통으로 나타나는 엔티티
    entity_tu_count = Counter()
    for tu_id, entities in tu_entities_map.items():
        for ent in entities:
            if ent != entity_name:
                entity_tu_count[ent] += 1
    
    cross_references = [
        {'entity': ent, 'text_unit_count': count}
        for ent, count in entity_tu_count.most_common(5)
        if count > 1
    ]
    
    # 4) Unique aspects: 각 텍스트 유닛의 고유한 측면
    unique_aspects = []
    unique_triples_by_tu = []  # 수정 5: 고유 트리플 추가
    tus = list(tu_entities_map.keys())
    
    for i, tu_id in enumerate(tus[:3]):  # 상위 3개만
        # 다른 텍스트 유닛들의 엔티티/관계/트리플 집합
        other_entities = set()
        other_reltypes = set()
        other_triples = set()
        
        for j, other_tu in enumerate(tus):
            if i != j:
                other_entities.update(tu_entities_map[other_tu])
                other_reltypes.update(tu_reltype_map[other_tu])
                other_triples.update(tu_triples_map[other_tu])
        
        # 고유한 것들
        unique_entities = tu_entities_map[tu_id] - other_entities
        unique_reltypes = tu_reltype_map[tu_id] - other_reltypes
        unique_triples = tu_triples_map[tu_id] - other_triples
        
        if unique_entities or unique_reltypes:
            unique_aspects.append({
                'text_unit_id': tu_id,
                'unique_entities': sorted(list(unique_entities))[:3],
                'unique_relation_types': sorted(list(unique_reltypes))[:3]
            })
        
        # 수정 5: 고유 트리플 추가
        if unique_triples:
            unique_triples_by_tu.append({
                'text_unit_id': tu_id,
                'unique_triples': [
                    {'s': s, 'p': p, 'o': o} 
                    for s, p, o in sorted(list(unique_triples))[:3]
                ]
            })
    
    # 5) Common themes: 공통 관계 타입
    rel_type_count = Counter()
    for tu_id, rel_types in tu_reltype_map.items():
        for rt in rel_types:
            rel_type_count[rt] += 1
    
    common_themes = [
        {'relation_type': rt, 'frequency': count}
        for rt, count in rel_type_count.most_common(3)
        if count > 1
    ]
    
    # 6) Common triples: 여러 텍스트 유닛에 걸쳐 나타나는 트리플
    # 수정 3: 텍스트 유닛별로 중복 제거 후 카운트
    triple_tu_count = Counter()
    for tu_id in text_units:
        seen_in_tu = set()  # 이 텍스트 유닛에서 본 트리플
        
        for triple in TU_TO_TRIPLES.get(tu_id, []):
            if triple["s"] == entity_name or triple["o"] == entity_name:
                key = (triple["s"], triple["p"], triple["o"])
                seen_in_tu.add(key)
        
        # 각 고유 트리플을 한 번씩만 카운트
        for key in seen_in_tu:
            triple_tu_count[key] += 1
    
    common_triples = [
        {'s': s, 'p': p, 'o': o, 'text_unit_count': count}
        for (s, p, o), count in triple_tu_count.most_common(3)
        if count > 1
    ]
    
    # 7) Evidence coverage
    evidence_coverage = {
        'total_candidate_text_units': len(candidate_tus),
        'analyzed_text_units': len(text_units),
        'total_triples_analyzed': sum(len(TU_TO_TRIPLES.get(tu, [])) for tu in text_units),
        'anchor_triples_count': sum(info['triple_count'] for info in info_by_tu)
    }
    
    return {
        'entity': entity_name,
        'text_unit_count': len(text_units),
        'text_units': text_units,
        'info_by_text_unit': info_by_tu,
        'relationships_by_text_unit': relationships_by_tu,
        'cross_references': cross_references,
        'unique_aspects': unique_aspects,
        'unique_triples_by_text_unit': unique_triples_by_tu,  # 수정 5: 추가
        'common_themes': common_themes,
        'common_triples': common_triples,
        'evidence_coverage': evidence_coverage
    }

# 테스트
print("\n=== Cross-context Integration 정답 구조 계산 테스트 (최종 개선 버전) ===")
if all_questions.get('cross_context'):
    test_q = all_questions['cross_context'][0]
    print(f"질문: {test_q['question'][:100]}...")
    
    # 실제로는 질문 생성시 저장된 pattern 사용
    pattern_data = {
        'entity': test_q.get('entity', '고추')  # 임시로 고추 사용
    }
    
    answer_struct = compute_cross_context_answer_struct(test_q['question'], pattern_data)
    print(f"\n계산된 정답 구조:")
    print(f"- 중심 엔티티: {answer_struct['entity']}")
    print(f"- 관련 텍스트 유닛: {answer_struct['text_unit_count']}개 (anchor triple 기준 상위)")
    print(f"- Cross-references: {len(answer_struct['cross_references'])}개")
    if answer_struct['cross_references']:
        print(f"  - 최다 공통 엔티티: {answer_struct['cross_references'][0]['entity']} ({answer_struct['cross_references'][0]['text_unit_count']}개 텍스트)")
    print(f"- Unique aspects: {len(answer_struct['unique_aspects'])}개")
    print(f"- Unique triples by TU: {len(answer_struct['unique_triples_by_text_unit'])}개")
    print(f"- Common themes: {len(answer_struct['common_themes'])}개")
    print(f"- Common triples: {len(answer_struct['common_triples'])}개")
    print(f"- Coverage: {answer_struct['evidence_coverage']['analyzed_text_units']}/{answer_struct['evidence_coverage']['total_candidate_text_units']} 텍스트 유닛")
    print(f"  - 분석된 anchor 트리플: {answer_struct['evidence_coverage']['anchor_triples_count']}개")

## Cell 40: Implicit Relationships 정답 구조 계산

직접 연결되지 않은 엔티티 간의 암묵적 관계를 공통 이웃을 통해 찾는 질문의 정답 구조를 계산합니다.

In [ ]:
import math

# Helper 함수들 추가
def _edge_attrs_iter(G, u, v):
    """(u,v) 사이 엣지 attrs를 안전하게 반복"""
    data = G.get_edge_data(u, v)
    if not data:
        return []
    if G.is_multigraph():
        return list(data.items())  # (key, attrdict)
    return [(None, data)]  # (None, attrdict)

def _tu_ids_str(attr, top_k=3):
    """text unit IDs를 str로 강제 변환"""
    ids = collect_evidence_text_unit_ids(attr, top_k=top_k) or []
    return [str(x) for x in ids]

def _extract_edges(G, u, v, evidence_ids, top_k=3):
    """u->v 방향 엣지들 수집 (directed면 u->v만, undirected면 양방향 동일)"""
    out = []
    if not G.has_edge(u, v):
        return out
    
    for k, attr in _edge_attrs_iter(G, u, v):
        rel = _get_relation_type(attr) or "related_to"
        tu_ids = _tu_ids_str(attr, top_k=top_k)
        evidence_ids.update(tu_ids)
        
        out.append({
            "from": u,
            "to": v,
            "relation": rel,
            "weight": float(attr.get("weight", 0.0) or 0.0),
            "text_unit_ids": tu_ids
        })
    return out

def compute_implicit_relationships_answer_struct(question: str, pattern_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Implicit Relationships 질문에 대한 정답 구조 계산 (최종 개선 버전)
    
    직접 연결되지 않은 두 엔티티 간의 암묵적 관계를 공통 이웃을 통해 분석
    
    Returns:
        answer_struct: {
            'entity1': 첫 번째 엔티티,
            'entity2': 두 번째 엔티티,
            'is_valid': 유효한 샘플 여부,
            'filter_reason': 필터링 이유,
            'has_direct_connection': 직접 연결 여부,
            'common_neighbors': 공통 이웃 리스트,
            'common_neighbor_count': 공통 이웃 수,
            'mediator': 가장 중요한 매개 엔티티,
            'mediator_ranking': 매개 엔티티 순위,
            'two_hop_motifs': 2-hop 연결 패턴,
            'evidence_paths': 증거 경로 리스트,
            'metrics': 관계 강도 메트릭,
            'evidence_text_unit_ids': 정렬된 text unit IDs
        }
    """
    # Pattern에서 두 엔티티 추출
    entity1 = pattern_data.get('entity1')
    entity2 = pattern_data.get('entity2')
    
    base_empty = {
        'entity1': entity1,
        'entity2': entity2,
        'is_valid': False,
        'filter_reason': 'missing_or_unknown_entity',
        'has_direct_connection': False,
        'common_neighbors': [],
        'common_neighbor_count': 0,
        'mediator': None,
        'mediator_ranking': [],
        'two_hop_motifs': [],
        'evidence_paths': [],
        'metrics': {},
        'evidence_text_unit_ids': []
    }
    
    if not entity1 or not entity2:
        return base_empty
    if entity1 not in G or entity2 not in G:
        return base_empty
    
    # 1) 직접 연결 확인 (양방향)
    has_direct = G.has_edge(entity1, entity2) or G.has_edge(entity2, entity1)
    
    # 직접 연결이 있으면 implicit relationships에 적합하지 않음
    if has_direct:
        return {
            **base_empty,
            'is_valid': False,
            'filter_reason': 'direct_edge_exists',
            'has_direct_connection': True
        }
    
    # 2) 공통 이웃 찾기 (underlying undirected view)
    if G.is_directed():
        neighbors1 = set(G.successors(entity1)) | set(G.predecessors(entity1))
        neighbors2 = set(G.successors(entity2)) | set(G.predecessors(entity2))
    else:
        neighbors1 = set(G.neighbors(entity1))
        neighbors2 = set(G.neighbors(entity2))
    
    common_neighbors = neighbors1 & neighbors2
    
    # 공통 이웃이 없으면 2-hop 경로 없음
    if not common_neighbors:
        return {
            **base_empty,
            'is_valid': False,
            'filter_reason': 'no_common_neighbor_two_hop',
            'has_direct_connection': False
        }
    
    # 3) Mediator 선정: Adamic-Adar 점수 + degree centrality
    G_undirected = G.to_undirected() if G.is_directed() else G
    degree_cent = nx.degree_centrality(G_undirected)
    
    # Mediator 점수 계산
    mediator_scores = []
    for w in common_neighbors:
        deg = G_undirected.degree(w)
        # Adamic-Adar local contribution: 1/log(degree)
        aa_score = 0.0 if deg <= 1 else 1.0 / math.log(deg)
        
        mediator_scores.append({
            'entity': w,
            'aa_score': aa_score,
            'degree_centrality': degree_cent.get(w, 0.0),
            'degree': deg
        })
    
    # 정렬: AA 점수 내림차순, degree centrality 내림차순, 이름 오름차순 (str 강제)
    mediator_scores.sort(key=lambda x: (-x['aa_score'], -x['degree_centrality'], str(x['entity'])))
    
    mediator = mediator_scores[0]['entity'] if mediator_scores else None
    mediator_ranking = [
        {
            'common_entity': m['entity'],
            'aa_score': round(m['aa_score'], 6),
            'degree_centrality': round(m['degree_centrality'], 6)
        }
        for m in mediator_scores[:10]
    ]
    
    common_neighbors_sorted = [m['entity'] for m in mediator_scores[:10]]
    
    # 4) Two-hop motifs 분석 (유효 motif가 K개 모일 때까지 mediator 후보를 더 탐색)
    evidence_ids = set()
    motifs = []
    
    for med_info in mediator_scores[:10]:  # 후보는 10개까지 본다
        w = med_info['entity']
        motif = {
            'path': [entity1, w, entity2],
            'mediator': w,
            'hop1_edges': [],
            'hop2_edges': [],
            'motif_type': 'two_hop'
        }
        
        # hop1: entity1 <-> w  (방향 모두 수집)
        motif['hop1_edges'].extend(_extract_edges(G, entity1, w, evidence_ids, top_k=3))
        if G.is_directed():
            motif['hop1_edges'].extend(_extract_edges(G, w, entity1, evidence_ids, top_k=3))
        
        # hop2: w <-> entity2  (방향 모두 수집)
        motif['hop2_edges'].extend(_extract_edges(G, w, entity2, evidence_ids, top_k=3))
        if G.is_directed():
            motif['hop2_edges'].extend(_extract_edges(G, entity2, w, evidence_ids, top_k=3))
        
        # 유효한 motif인지 확인 (양쪽 hop 모두 있어야 함)
        if motif['hop1_edges'] and motif['hop2_edges']:
            motifs.append(motif)
        
        # 충분히 모이면 중단 (결정성 유지)
        if len(motifs) >= 5:
            break
    
    # 5) Evidence paths: 상위 3개 motif의 구조화된 경로
    evidence_paths = []
    for motif in motifs[:3]:
        path_obj = {
            'path': motif['path'],
            'support': {
                'hop1': [
                    {
                        's': e['from'], 
                        'p': e['relation'], 
                        'o': e['to'],
                        'text_unit_ids': e['text_unit_ids']
                    }
                    for e in motif['hop1_edges']
                ],
                'hop2': [
                    {
                        's': e['from'], 
                        'p': e['relation'], 
                        'o': e['to'],
                        'text_unit_ids': e['text_unit_ids']
                    }
                    for e in motif['hop2_edges']
                ]
            }
        }
        evidence_paths.append(path_obj)
    
    # 6) Metrics 계산
    # Jaccard similarity
    jaccard = len(common_neighbors) / len(neighbors1 | neighbors2) if (neighbors1 | neighbors2) else 0.0
    
    # Adamic-Adar sum over common neighbors  
    aa_sum_over_common_neighbors = sum(m['aa_score'] for m in mediator_scores)
    
    # Implicit strength (0-1 범위)
    cn_score = min(len(common_neighbors) / 10, 1.0)
    top_aa = mediator_scores[0]['aa_score'] if mediator_scores else 0.0
    aa_norm = min(top_aa / 2.0, 1.0)  # 도메인에 따라 조정
    strength = round((cn_score + aa_norm + min(jaccard * 2, 1.0)) / 3, 3)
    
    # evidence 정렬 (타입 안전성)
    evidence_ids_sorted = sorted(set(map(str, evidence_ids)))[:10]
    
    # 최종 결과
    result = {
        'entity1': entity1,
        'entity2': entity2,
        'is_valid': True,
        'filter_reason': None,
        'has_direct_connection': has_direct,
        'common_neighbors': common_neighbors_sorted,
        'common_neighbor_count': len(common_neighbors),
        'mediator': mediator,
        'mediator_ranking': mediator_ranking,
        'two_hop_motifs': motifs,
        'evidence_paths': evidence_paths,
        'metrics': {
            'jaccard_neighbors': round(jaccard, 6),
            'aa_sum_over_common_neighbors': round(aa_sum_over_common_neighbors, 6),
            'implicit_strength': strength
        },
        'evidence_text_unit_ids': evidence_ids_sorted
    }
    
    # motif가 하나도 없으면 유효하지 않음
    if not motifs:
        result['is_valid'] = False
        result['filter_reason'] = 'no_valid_two_hop_edges'
        result['evidence_paths'] = []
        result['evidence_text_unit_ids'] = []
    
    return result

# 테스트
print("=== Implicit Relationships 정답 구조 계산 테스트 (최종 개선 버전) ===")
if all_questions.get('implicit_relationships'):
    test_q = all_questions['implicit_relationships'][0]
    print(f"질문: {test_q['question'][:100]}...")
    
    # 임시 pattern data - 실제로는 질문 생성시 저장된 값 사용
    pattern_data = {
        'entity1': test_q.get('entity1', '고추'),
        'entity2': test_q.get('entity2', '토마토')
    }
    
    answer_struct = compute_implicit_relationships_answer_struct(test_q['question'], pattern_data)
    print(f"\n계산된 정답 구조:")
    print(f"- Entity 1: {answer_struct['entity1']}")
    print(f"- Entity 2: {answer_struct['entity2']}")
    print(f"- 유효성: {answer_struct['is_valid']}")
    if not answer_struct['is_valid']:
        print(f"- 필터 이유: {answer_struct['filter_reason']}")
    print(f"- 직접 연결: {answer_struct['has_direct_connection']}")
    print(f"- 공통 이웃 수: {answer_struct['common_neighbor_count']}")
    
    if answer_struct['mediator']:
        print(f"- 최고 중요도 매개자: {answer_struct['mediator']}")
        if answer_struct['mediator_ranking']:
            top = answer_struct['mediator_ranking'][0]
            print(f"  - AA score: {top['aa_score']}, Centrality: {top['degree_centrality']}")
    
    print(f"- 2-hop motifs: {len(answer_struct['two_hop_motifs'])}개")
    print(f"- Evidence 경로: {len(answer_struct['evidence_paths'])}개")
    print(f"- Metrics:")
    for k, v in answer_struct['metrics'].items():
        print(f"  - {k}: {v}")
    print(f"- Evidence IDs: {len(answer_struct['evidence_text_unit_ids'])}개")

## Cell 41: Causal Chains 정답 구조 계산

인과 관계 체인 질문에 대해 그래프에서 실제 인과 경로를 찾고 관련 정보를 추출합니다.

In [ ]:
# Causal Chains 정답 구조 계산 함수 (개선 버전)
def choose_best_causal_edge(edge_attrs, causal_relations):
    """인과 관계 엣지 중 가장 적합한 것을 결정적으로 선택"""
    scored = []
    for attr in edge_attrs:
        rel = _get_relation_type(attr) or ""
        desc = str(attr.get("description", "")).lower()
        tu_ids = _tu_ids_str(attr, top_k=3)
        weight = float(attr.get("weight", 0) or 0)
        
        # 인과 키워드 체크
        causal_keywords = ['cause', 'caus', 'lead', 'result', 'trigger', 
                          'produce', 'affect', 'influence', '원인', '결과', 
                          '초래', '영향', '유발', '야기']
        has_causal_kw = any(kw in desc for kw in causal_keywords)
        is_causal_rel = rel in causal_relations
        
        # 점수 계산: 명시적 인과관계 > 키워드 포함 > evidence 수 > weight
        score = (
            (10 if is_causal_rel else 0) +
            (5 if has_causal_kw else 0) +
            min(len(tu_ids), 3) +
            weight
        )
        
        # 최소 점수 기준 (causal relation이거나 keyword가 있어야 함)
        if is_causal_rel or has_causal_kw:
            scored.append((score, rel, desc, weight, tu_ids, attr))
    
    # 점수 내림차순, relation 이름, description 순으로 정렬 (결정성)
    if scored:
        scored.sort(key=lambda x: (-x[0], x[1], x[2]))
        return scored[0][-1]  # attr 반환
    return None

def compute_causal_chains_answer_struct(question: str, pattern_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Causal Chains 질문에 대한 정답 구조 계산 (개선 버전)
    
    인과 관계 체인을 그래프에서 추출하고 단계별 영향을 분석
    """
    # Pattern에서 정보 추출
    chain_type = pattern_data.get('chain_type', 'linear')
    start_event = pattern_data.get('start_event')
    terminal_effects = pattern_data.get('terminal_effects', [])
    
    empty_result = {
        'chain_type': chain_type,
        'start_event': start_event,
        'terminal_effects': terminal_effects,
        'causal_paths': [],
        'chain_length': 0,
        'intermediate_steps': [],
        'branch_points': [],
        'causal_relationships': [],
        'evidence_by_step': [],
        'causal_strength_metrics': {'path_count': 0},
        'evidence_text_unit_ids': []
    }
    
    if not start_event or start_event not in G:
        return empty_result
    
    # 인과 관계를 나타내는 관계 타입들 (related_to 제외!)
    causal_relations = {
        'causes', 'leads_to', 'results_in', 'triggers', 'produces',
        'affects', 'influences', 'causes_increase', 'causes_decrease',
        '원인', '유발', '초래', '영향', '야기', '결과'
    }
    
    evidence_ids = set()
    all_causal_edges = []  # 전체 인과 엣지 수집
    
    # 1) BFS로 인과 체인 탐색 (최대 5 hop)
    all_paths = []
    max_depth = 5
    
    from collections import deque
    queue = deque([(start_event, [start_event], [])])  # (current, path, edges)
    visited_paths = set()
    
    while queue:
        current, path, edges = queue.popleft()
        
        # 경로 길이 제한
        if len(path) > max_depth + 1:  # path includes start node
            continue
            
        # 경로 키 생성 (중복 제거용)
        path_key = tuple(path)
        if path_key in visited_paths:
            continue
        visited_paths.add(path_key)
        
        # terminal effect에 도달했으면 경로 저장
        if terminal_effects and current in terminal_effects and len(path) > 1:
            # 경로에 최소 1개 이상의 인과 엣지가 있는지 확인
            has_causal = any(e.get('is_causal', False) for e in edges)
            if has_causal:
                all_paths.append({
                    'path': path,
                    'edges': edges,
                    'length': len(path) - 1
                })
        
        # terminal_effects가 비어있고 더 이상 갈 곳이 없으면 leaf로 간주
        if not terminal_effects:
            # 현재 노드에서 나가는 인과 엣지가 없으면 terminal로 간주
            has_outgoing = False
            if G.is_directed():
                for neighbor in G.successors(current):
                    attrs = _edge_attr_dicts(G, current, neighbor)
                    if any(choose_best_causal_edge(attrs, causal_relations) is not None for _ in [0]):
                        has_outgoing = True
                        break
            else:
                for neighbor in G.neighbors(current):
                    if neighbor not in path:
                        attrs = _edge_attr_dicts(G, current, neighbor)
                        if any(choose_best_causal_edge(attrs, causal_relations) is not None for _ in [0]):
                            has_outgoing = True
                            break
            
            if not has_outgoing and len(path) > 1:
                has_causal = any(e.get('is_causal', False) for e in edges)
                if has_causal:
                    all_paths.append({
                        'path': path,
                        'edges': edges,
                        'length': len(path) - 1,
                        'is_leaf': True
                    })
        
        # 다음 노드 탐색 (directed면 successors만, undirected면 neighbors)
        if G.is_directed():
            neighbors = list(G.successors(current))
        else:
            neighbors = list(G.neighbors(current))
        
        for neighbor in neighbors:
            if neighbor not in path:  # 사이클 방지
                # 인과 관계 엣지 확인
                edge_attrs = _edge_attr_dicts(G, current, neighbor)
                best_attr = choose_best_causal_edge(edge_attrs, causal_relations)
                
                if best_attr:
                    rel = _get_relation_type(best_attr) or "unknown"
                    tu_ids = _tu_ids_str(best_attr, top_k=3)
                    
                    new_edge = {
                        'from': current,
                        'to': neighbor,
                        'relation': rel,
                        'weight': float(best_attr.get('weight', 0)),
                        'description': best_attr.get('description', ''),
                        'text_unit_ids': tu_ids,
                        'is_causal': True
                    }
                    
                    all_causal_edges.append(new_edge)
                    queue.append((neighbor, path + [neighbor], edges + [new_edge]))
    
    # 경로가 없으면 빈 결과 반환
    if not all_paths:
        return empty_result
    
    # 2) 경로 정렬 및 선택 (길이순, weight 합순)
    for path_info in all_paths:
        path_info['total_weight'] = sum(e['weight'] for e in path_info['edges'])
        path_info['causal_edge_count'] = sum(1 for e in path_info['edges'] if e.get('is_causal'))
    
    # 정렬: 인과 엣지 수 > 길이 > weight 합
    all_paths.sort(key=lambda x: (-x['causal_edge_count'], -x['length'], -x['total_weight']))
    
    # 3) Causal paths 구성 (상위 5개)
    causal_paths = []
    causal_relationships = []
    
    for path_info in all_paths[:5]:
        path = path_info['path']
        edges = path_info['edges']
        
        # 경로별 단계 구성
        steps = []
        for i, edge in enumerate(edges):
            evidence_ids.update(edge['text_unit_ids'])
            
            steps.append({
                'step': i + 1,
                'from': edge['from'],
                'to': edge['to'],
                'relation': edge['relation'],
                'description': edge['description'],
                'text_unit_ids': edge['text_unit_ids']
            })
            
            # 전체 인과 관계 리스트에 추가
            causal_relationships.append({
                'source': edge['from'],
                'relation': edge['relation'],
                'target': edge['to'],
                'weight': edge['weight'],
                'in_path': tuple(path)
            })
        
        causal_paths.append({
            'path': path,
            'steps': steps,
            'length': len(edges),
            'total_weight': path_info['total_weight'],
            'is_leaf_path': path_info.get('is_leaf', False)
        })
    
    # 4) Causal subgraph 구성 (중간 단계와 분기점 분석용)
    causal_subgraph = nx.MultiDiGraph() if G.is_multigraph() else nx.DiGraph()
    for edge in all_causal_edges:
        causal_subgraph.add_edge(
            edge['from'], edge['to'],
            relation=edge['relation'],
            weight=edge['weight']
        )
    
    # 5) 중간 단계 분석 (causal subgraph 기준)
    intermediate_nodes = set()
    for path_info in all_paths:
        intermediate_nodes.update(path_info['path'][1:-1])
    
    intermediate_steps = []
    for node in sorted(intermediate_nodes)[:10]:
        # causal subgraph에서의 degree
        if node in causal_subgraph:
            in_degree = causal_subgraph.in_degree(node)
            out_degree = causal_subgraph.out_degree(node)
        else:
            in_degree = out_degree = 0
        
        # 노드가 몇 개의 경로에 등장하는지
        path_count = sum(1 for p in all_paths if node in p['path'][1:-1])
        
        intermediate_steps.append({
            'entity': node,
            'causal_in_degree': in_degree,
            'causal_out_degree': out_degree,
            'path_appearances': path_count,
            'role': 'amplifier' if out_degree > in_degree else 'bottleneck'
        })
    
    # 6) Branch points 분석 (실제 경로에서의 분기)
    branch_points = []
    if chain_type == 'branching':
        # 경로들에서 실제로 분기가 일어난 지점 찾기
        node_next_hops = defaultdict(set)
        for path_info in all_paths:
            path = path_info['path']
            for i in range(len(path) - 1):
                node_next_hops[path[i]].add(path[i + 1])
        
        # 2개 이상의 다른 다음 노드를 가진 노드들
        for node, next_nodes in node_next_hops.items():
            if len(next_nodes) > 1:
                branch_points.append({
                    'node': node,
                    'branches': sorted(list(next_nodes))[:5],
                    'branch_count': len(next_nodes),
                    'causal_out_degree': causal_subgraph.out_degree(node) if node in causal_subgraph else 0
                })
        
        # 분기 수 기준으로 정렬
        branch_points.sort(key=lambda x: -x['branch_count'])
        branch_points = branch_points[:5]
    
    # 7) Evidence by step (주 경로 기준만)
    evidence_by_step = []
    main_evidence_ids = set()
    
    if causal_paths:
        main_path = causal_paths[0]
        for i, step in enumerate(main_path['steps']):
            main_evidence_ids.update(step['text_unit_ids'])
            evidence_by_step.append({
                'step_number': i + 1,
                'transition': f"{step['from']} → {step['to']}",
                'relation': step['relation'],
                'text_unit_ids': step['text_unit_ids']
            })
    
    # 8) Causal strength metrics
    max_chain_length = max(p['length'] for p in all_paths) if all_paths else 0
    
    # 경로 다양성
    unique_intermediate_sets = set()
    for path_info in all_paths:
        intermediates = tuple(sorted(path_info['path'][1:-1]))
        unique_intermediate_sets.add(intermediates)
    
    path_diversity = len(unique_intermediate_sets) / len(all_paths) if all_paths else 0
    avg_path_weight = sum(p['total_weight'] for p in all_paths) / len(all_paths) if all_paths else 0
    
    # Terminal coverage 계산
    if terminal_effects:
        reached_terminals = set(p['path'][-1] for p in all_paths if p['path'][-1] in terminal_effects)
        terminal_coverage = len(reached_terminals) / len(terminal_effects)
    else:
        # terminal이 없으면 leaf 도달률로 대체
        terminal_coverage = 1.0 if any(p.get('is_leaf', False) for p in all_paths) else 0.0
    
    causal_strength_metrics = {
        'path_count': len(all_paths),
        'max_chain_length': max_chain_length,
        'path_diversity': round(path_diversity, 3),
        'average_path_weight': round(avg_path_weight, 3),
        'terminal_coverage': round(terminal_coverage, 3),
        'causal_edge_count': len(all_causal_edges)
    }
    
    # evidence는 주 경로 기준으로만 (정보 누수 방지)
    evidence_ids_sorted = sorted(set(map(str, main_evidence_ids)))[:15]
    
    return {
        'chain_type': chain_type,
        'start_event': start_event,
        'terminal_effects': terminal_effects[:10] if terminal_effects else ["(자연 종료)"],
        'causal_paths': causal_paths,
        'chain_length': max_chain_length,
        'intermediate_steps': intermediate_steps,
        'branch_points': branch_points,
        'causal_relationships': causal_relationships[:20],
        'evidence_by_step': evidence_by_step,
        'causal_strength_metrics': causal_strength_metrics,
        'evidence_text_unit_ids': evidence_ids_sorted
    }

# 테스트
print("=== Causal Chains 정답 구조 계산 테스트 (개선 버전) ===")
if all_questions.get('causal_chains'):
    test_q = all_questions['causal_chains'][0]
    print(f"질문: {test_q['question'][:100]}...")
    
    # 실제 패턴 데이터 찾기
    # 먼저 실제 그래프에서 인과 관계가 있는 노드들을 탐색
    print("\n그래프에서 인과 관계 탐색 중...")
    
    # 인과 관계 키워드를 포함하는 엣지들 찾기
    causal_edges = []
    causal_keywords = ['cause', 'caus', 'lead', 'result', 'trigger', 
                      'produce', 'affect', 'influence', '원인', '결과', 
                      '초래', '영향', '유발', '야기']
    
    sample_count = 0
    for u, v, data in G.edges(data=True):
        desc = str(data.get('description', '')).lower()
        rel = data.get('relation', data.get('type', ''))
        
        if any(kw in desc for kw in causal_keywords) or rel in ['causes', 'leads_to', 'results_in']:
            causal_edges.append((u, v, rel, desc[:50]))
            sample_count += 1
            if sample_count >= 5:  # 샘플 5개만
                break
    
    print(f"발견된 인과 관계 샘플:")
    for u, v, rel, desc in causal_edges:
        print(f"  {u} --[{rel}]--> {v}: {desc}...")
    
    # 테스트를 위한 더 현실적인 pattern data
    if causal_edges:
        # 첫 번째 인과 엣지의 source를 시작점으로
        test_start = causal_edges[0][0]
        pattern_data = {
            'chain_type': 'linear',
            'start_event': test_start,
            'terminal_effects': []  # 자연 종료 테스트
        }
    else:
        # 대체 테스트 데이터
        pattern_data = {
            'chain_type': 'linear',
            'start_event': '고추',  # 실제 존재하는 노드
            'terminal_effects': []
        }
    
    print(f"\n테스트 패턴: start_event={pattern_data['start_event']}")
    
    answer_struct = compute_causal_chains_answer_struct(test_q['question'], pattern_data)
    print(f"\n계산된 정답 구조:")
    print(f"- Chain type: {answer_struct['chain_type']}")
    print(f"- 시작 이벤트: {answer_struct['start_event']}")
    print(f"- 최종 결과: {answer_struct['terminal_effects']}")
    print(f"- 인과 경로: {len(answer_struct['causal_paths'])}개 발견")
    
    if answer_struct['causal_paths']:
        main_path = answer_struct['causal_paths'][0]
        print(f"\n주 경로:")
        print(f"  경로: {' → '.join(main_path['path'])}")
        print(f"  길이: {main_path['length']} steps")
        print(f"  총 weight: {main_path['total_weight']:.3f}")
        print(f"  Leaf path: {main_path.get('is_leaf_path', False)}")
        
        print(f"\n  단계별 상세:")
        for step in main_path['steps'][:3]:  # 처음 3단계만
            print(f"    Step {step['step']}: {step['from']} --[{step['relation']}]--> {step['to']}")
    
    print(f"\n- 중간 단계: {len(answer_struct['intermediate_steps'])}개")
    if answer_struct['intermediate_steps']:
        for node_info in answer_struct['intermediate_steps'][:3]:
            print(f"  - {node_info['entity']}: in={node_info['causal_in_degree']}, out={node_info['causal_out_degree']}, role={node_info['role']}")
    
    print(f"\n- 분기점: {len(answer_struct['branch_points'])}개")
    print(f"\n- Metrics:")
    for k, v in answer_struct['causal_strength_metrics'].items():
        print(f"  - {k}: {v}")
    print(f"\n- Evidence IDs: {len(answer_struct['evidence_text_unit_ids'])}개 (주 경로 기준)")

## Cell 42: LLM 기반 자연어 표현 시스템 (유효성 검증 포함)

답변 구조를 자연어로 변환하고, 논리적 타당성을 검증하여 부적절한 QA 쌍을 필터링합니다.

In [ ]:
# LLM 기반 자연어 표현 및 유효성 검증 시스템 (최종 개선 버전)
import json
import re
from typing import Dict, Any, List, Tuple, Set

# ----------------------------
# Utilities
# ----------------------------

def _safe_str(x) -> str:
    return "" if x is None else str(x)

def _truncate(text: str, n: int) -> str:
    if not text:
        return ""
    return text if len(text) <= n else text[:n] + "…"

def _normalize_ws(s: str) -> str:
    """공백 정규화"""
    return re.sub(r"\s+", " ", _safe_str(s)).strip()

def format_evidence_texts(
    text_unit_ids: List[str],
    evidence_texts: Dict[str, str],
    max_texts: int = 6,
    max_chars_each: int = 260
) -> List[Dict[str, str]]:
    """Evidence를 리스트로 반환 (LLM이 인용하기 쉽게)"""
    out = []
    for tu_id in (text_unit_ids or [])[:max_texts]:
        tu = str(tu_id)
        if tu in evidence_texts and evidence_texts[tu]:
            out.append({
                "text_unit_id": tu,
                "text": _truncate(evidence_texts[tu], max_chars_each)
            })
    return out

def summarize_struct_for_validation(qa_type: str, answer_struct: Dict[str, Any]) -> Dict[str, Any]:
    """검증용으로 최소/필수 구조만 요약"""
    if qa_type == "multi_hop":
        return {
            "path_exists": bool(answer_struct.get("path_exists", False)),
            "path": answer_struct.get("path", [])[:8],
            "edges": answer_struct.get("edges", [])[:8],
            "hop_count": int(answer_struct.get("hop_count", 0) or 0),
        }
        
    if qa_type == "community_synthesis":
        return {
            "community_id": answer_struct.get("community_id"),
            "entity_count": int(answer_struct.get("entity_count", 0) or 0),
            "key_entities": answer_struct.get("key_entities", [])[:8],
            "top_relation_types": answer_struct.get("top_relation_types", [])[:3],
            "representative_triples": answer_struct.get("representative_triples", [])[:5],
            "density": answer_struct.get("density", 0.0),
        }
        
    if qa_type == "cross_context":
        return {
            "entity": answer_struct.get("entity"),
            "text_units": answer_struct.get("text_units", [])[:10],
            "common_themes": answer_struct.get("common_themes", [])[:5],
            "unique_aspects": answer_struct.get("unique_aspects", [])[:5],
            "common_triples": answer_struct.get("common_triples", [])[:5],
        }
        
    if qa_type == "implicit_relationships":
        return {
            "entity1": answer_struct.get("entity1"),
            "entity2": answer_struct.get("entity2"),
            "is_valid_struct": bool(answer_struct.get("is_valid", True)),
            "has_direct_connection": bool(answer_struct.get("has_direct_connection", False)),
            "mediator": answer_struct.get("mediator"),
            "common_neighbors": answer_struct.get("common_neighbors", [])[:10],
            "evidence_paths": answer_struct.get("evidence_paths", [])[:3],
            "metrics": answer_struct.get("metrics", {}),
        }
        
    if qa_type == "causal_chains":
        # 인과 검증 강화를 위해 step별 relation 포함
        step_rels = []
        for s in (answer_struct.get("evidence_by_step", []) or [])[:8]:
            step_rels.append({
                "transition": s.get("transition"),
                "relation": s.get("relation"),
                "text_unit_ids": s.get("text_unit_ids", [])[:3],
            })
            
        return {
            "chain_type": answer_struct.get("chain_type", "linear"),
            "start_event": answer_struct.get("start_event"),
            "terminal_effects": answer_struct.get("terminal_effects", [])[:10],
            "causal_paths": [
                {"path": p.get("path", [])[:10], "length": p.get("length")}
                for p in (answer_struct.get("causal_paths", [])[:3] or [])
            ],
            "evidence_by_step": step_rels,
            "metrics": answer_struct.get("causal_strength_metrics", {}),
        }
    
    return {}

# ----------------------------
# Rule-based precheck (강화)
# ----------------------------

# 인과 관계 화이트리스트
CAUSAL_REL_WHITELIST = {
    'causes', 'leads_to', 'results_in', 'triggers', 'produces',
    'affects', 'influences', 'causes_increase', 'causes_decrease',
    '원인', '유발', '초래', '영향', '야기', '결과'
}

# 인과 관계 텍스트 패턴
CAUSAL_TEXT_PAT = re.compile(
    r"(cause|caus|lead\s*to|result\s*in|trigger|produce|affect|influence|"
    r"원인|때문|으로\s*인해|로\s*인해|유발|초래|영향|야기|결과)",
    re.IGNORECASE
)

# 서지/학술 패턴 (인과관계로 부적절한 케이스 필터링)
BIBLIO_PAT = re.compile(
    r"(journal|conference|proceedings|doi|issn|publisher|학술지|논문|저자|게재|발행)",
    re.IGNORECASE
)

def _evidence_has_causal_signal(evidence_items: List[Dict[str, str]]) -> bool:
    """Evidence에 인과 신호가 있는지 확인"""
    if not evidence_items:
        return False
    joined = "\n".join(_safe_str(x.get("text", "")) for x in evidence_items)
    # 서지 패턴이 있으면 인과로 보기 어려움
    if BIBLIO_PAT.search(joined):
        return False
    return bool(CAUSAL_TEXT_PAT.search(joined))

def rule_based_precheck(
    qa_type: str,
    question: str,
    answer_struct: Dict[str, Any],
    evidence_items: List[Dict[str, str]] = None,
) -> Tuple[bool, str]:
    """LLM 호출 전 명백히 부적절한 샘플 필터링"""
    q = _normalize_ws(question)
    if len(q) < 5:
        return False, "question_too_short"

    if qa_type == "multi_hop":
        if not answer_struct.get("path_exists", False):
            return False, "no_path"
        if len(answer_struct.get("path", [])) < 2:
            return False, "path_too_short"
        if int(answer_struct.get("hop_count", 0) or 0) <= 0:
            return False, "hop_count_invalid"

    elif qa_type == "implicit_relationships":
        if answer_struct.get("is_valid", True) is False:
            return False, f"struct_invalid:{answer_struct.get('filter_reason', 'unknown')}"
        if answer_struct.get("has_direct_connection", False):
            return False, "direct_connection_exists"

    elif qa_type == "causal_chains":
        if len(answer_struct.get("causal_paths", []) or []) == 0:
            return False, "no_causal_paths"
            
        # 시작 이벤트가 서지/학술 관련이면 필터링
        start_event = str(answer_struct.get("start_event", "")).lower()
        if BIBLIO_PAT.search(start_event):
            return False, "causal_start_event_bibliographic"
            
        # 관계 타입 확인
        step_infos = answer_struct.get("evidence_by_step", []) or []
        rels = [_safe_str(s.get("relation", "")).strip() for s in step_infos]
        if rels:
            whitelist_count = sum(1 for r in rels if r in CAUSAL_REL_WHITELIST)
            unknown_count = sum(1 for r in rels if r.lower() in {"", "unknown", "related_to"})
            
            # 인과 관계가 하나도 없고 unknown이 대부분이면
            if whitelist_count == 0 and unknown_count >= len(rels) / 2:
                # Evidence에서도 인과 신호가 없으면 필터링
                if evidence_items and not _evidence_has_causal_signal(evidence_items):
                    return False, "no_causal_relations_or_evidence"

    return True, "ok"

def _build_evidence_index(evidence_items: List[Dict[str, str]]) -> Dict[str, str]:
    """Evidence text_unit_id to text 매핑"""
    return {
        str(x["text_unit_id"]): _safe_str(x.get("text", "")) 
        for x in (evidence_items or []) 
        if "text_unit_id" in x
    }

def local_claim_hard_check(
    claims: List[Dict[str, Any]], 
    evidence_items: List[Dict[str, str]]
) -> Tuple[bool, List[str]]:
    """Claims의 citations 유효성 하드 체크"""
    issues = []
    ev_index = _build_evidence_index(evidence_items)
    
    if not claims:
        return False, ["no_claims"]
        
    for i, claim in enumerate(claims):
        text = _normalize_ws(claim.get("text", ""))
        citations = claim.get("citations", []) or []
        citations = [str(x) for x in citations if _safe_str(x).strip()]
        
        if not text:
            issues.append(f"claim_{i}_empty_text")
            continue
            
        if not citations:
            issues.append(f"claim_{i}_no_citations")
            continue
            
        # 존재하지 않는 citation 체크
        bad_citations = [cid for cid in citations if cid not in ev_index]
        if bad_citations:
            issues.append(f"claim_{i}_invalid_citations:{bad_citations}")
    
    # 치명적 오류가 있으면 False
    fatal_issues = [x for x in issues if any(
        keyword in x for keyword in ["no_citations", "invalid_citations", "empty_text"]
    )]
    
    return len(fatal_issues) == 0, issues

# ----------------------------
# Generation (JSON output)
# ----------------------------

GEN_SYSTEM = (
    "당신은 한국 농업 도메인 지식그래프 QA 데이터셋 생성기입니다.\n"
    "반드시 제공된 Evidence(텍스트)와 구조 정보에서만 답을 작성하세요.\n"
    "Evidence에 없는 사실은 추론/상식으로 보완하지 말고 '근거 부족으로 알 수 없다'라고 쓰세요.\n"
    "출처는 text_unit_id로만 인용하세요.\n"
)

def build_generation_prompt(qa_type: str, question: str, struct_summary: Dict[str, Any],
                            evidence_items: List[Dict[str, str]]) -> str:
    """생성 프롬프트 구성"""
    # QA 타입별 추가 지시사항
    type_specific = ""
    if qa_type == "causal_chains":
        type_specific = "\n특히 인과관계는 '원인', '결과', '영향' 등의 표현을 Evidence에서 찾아 사용하세요."
    elif qa_type == "implicit_relationships":
        type_specific = "\n반드시 '직접적인 연결은 없지만' 형태로 시작하여 간접 관계를 설명하세요."
        
    return f"""
[질문 유형] {qa_type}
[질문] {question}

[구조 요약(JSON)]
{json.dumps(struct_summary, ensure_ascii=False)}

[Evidence 목록]
{json.dumps(evidence_items, ensure_ascii=False)}

[작업]
1) 위 Evidence와 구조 요약에 근거해 한국어 답변을 작성하세요.
2) 답변은 3~7문장 내로 작성하세요.
3) 답변에 포함된 각 문장(주장)은 반드시 citations로 근거(text_unit_id)를 달아야 합니다.
4) 구조 요약과 모순되는 내용(없는 노드/관계/단계 추가)은 금지합니다.{type_specific}

[반환 형식: JSON만]
{{
  "answer": "자연어 답변",
  "claims": [
    {{"text": "주장 1", "citations": ["<text_unit_id>", "..."]}},
    {{"text": "주장 2", "citations": ["..."]}}
  ],
  "coverage_note": "근거가 부족한 부분이 있으면 짧게(없으면 빈 문자열)"
}}
""".strip()

@retry_with_exponential_backoff
def generate_natural_language_answer(
    qa_type: str,
    question: str,
    answer_struct: Dict[str, Any],
    evidence_texts: Dict[str, str],
) -> Dict[str, Any]:
    """답변 생성 및 검증"""
    
    # Evidence 준비
    tu_ids = answer_struct.get("evidence_text_unit_ids", []) or []
    evidence_items = format_evidence_texts(tu_ids, evidence_texts, max_texts=6, max_chars_each=260)
    
    # 사전 체크
    ok, reason = rule_based_precheck(qa_type, question, answer_struct, evidence_items=evidence_items)
    if not ok:
        return {
            "answer": "",
            "claims": [],
            "is_valid": False,
            "validation_reason": f"precheck_failed:{reason}",
            "confidence": 0.0,
            "issues": [reason],
        }
    
    # 프롬프트 구성
    struct_summary = summarize_struct_for_validation(qa_type, answer_struct)
    prompt = build_generation_prompt(qa_type, question, struct_summary, evidence_items)
    
    # LLM 호출 (max_tokens -> max_completion_tokens로 변경)
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": GEN_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
        max_completion_tokens=900,  # max_tokens 대신 max_completion_tokens 사용
        response_format={"type": "json_object"},
    )
    
    # 응답 파싱
    try:
        gen = json.loads(resp.choices[0].message.content)
    except Exception:
        return {
            "answer": "",
            "claims": [],
            "is_valid": False,
            "validation_reason": "generation_json_parse_failed",
            "confidence": 0.0,
            "issues": ["generation_json_parse_failed"],
        }
    
    generated_answer = _normalize_ws(gen.get("answer", ""))
    claims = gen.get("claims", []) or []
    
    # 기본 품질 체크
    if len(generated_answer) < 20:
        return {
            "answer": generated_answer,
            "claims": claims,
            "is_valid": False,
            "validation_reason": "generation_too_short",
            "confidence": 0.0,
            "issues": ["generation_too_short"],
        }
    
    # Citations 하드 체크
    hard_ok, hard_issues = local_claim_hard_check(claims, evidence_items)
    if not hard_ok:
        return {
            "answer": generated_answer,
            "claims": claims,
            "is_valid": False,
            "validation_reason": "local_hard_check_failed",
            "confidence": 0.0,
            "issues": hard_issues,
        }
    
    # LLM 검증
    validation = validate_answer(
        qa_type=qa_type,
        question=question,
        answer=generated_answer,
        claims=claims,
        struct_summary=struct_summary,
        evidence_items=evidence_items,
    )
    
    # 타입별 추가 후처리
    if qa_type == "causal_chains" and validation["is_valid"]:
        # 인과 신호 최종 확인
        if not _evidence_has_causal_signal(evidence_items):
            validation = {
                "is_valid": False,
                "reason": "postcheck:no_causal_signal_in_evidence",
                "confidence": min(validation["confidence"], 0.4),
                "issues": validation.get("issues", []) + ["no_causal_signal"],
            }
            
    elif qa_type == "implicit_relationships" and validation["is_valid"]:
        # 답변에 "직접 연결 없음" 언급 확인
        if "직접" not in generated_answer:
            validation["issues"] = validation.get("issues", []) + ["missing_no_direct_mention"]
            validation["confidence"] = min(validation["confidence"], 0.8)
    
    return {
        "answer": generated_answer,
        "claims": claims,
        "is_valid": validation["is_valid"],
        "validation_reason": validation["reason"],
        "confidence": validation["confidence"],
        "issues": validation.get("issues", []),
    }

# ----------------------------
# Validation (Evidence-grounded)
# ----------------------------

VAL_SYSTEM = (
    "당신은 농업 지식그래프 QA 데이터셋의 검증자입니다.\n"
    "반드시 Evidence와 구조 요약만으로 판단하세요.\n"
    "답변의 각 claim이 Evidence에 의해 지지되는지, 구조 요약과 모순이 없는지 평가하세요.\n"
    "그럴듯함/상식은 금지합니다.\n"
)

def build_validation_prompt(qa_type: str, question: str, answer: str,
                            claims: List[Dict[str, Any]],
                            struct_summary: Dict[str, Any],
                            evidence_items: List[Dict[str, str]]) -> str:
    
    # 타입별 추가 검증 지시
    type_rules = ""
    if qa_type == "causal_chains":
        type_rules = "\n- 인과 체인 답변에는 Evidence에서 인과 표현(원인/결과/영향 등)이 확인되어야 함"
    elif qa_type == "implicit_relationships":
        type_rules = "\n- 암묵적 관계 답변에는 '직접 연결 없음' 언급이 필요함"
        
    return f"""
[질문 유형] {qa_type}
[질문] {question}

[답변]
{answer}

[Claims(JSON)]
{json.dumps(claims, ensure_ascii=False)}

[구조 요약(JSON)]
{json.dumps(struct_summary, ensure_ascii=False)}

[Evidence 목록]
{json.dumps(evidence_items, ensure_ascii=False)}

[검증 규칙]
1) 각 claim은 citations의 text_unit_id Evidence에 의해 직접 지지되어야 함
2) citations가 비어있거나 Evidence 목록에 없는 id면 그 claim은 실패
3) claim이 구조 요약과 모순되면 실패
4) 농업 상식으로 보완/추론한 내용은 실패{type_rules}

[반환 형식: JSON만]
{{
  "is_valid": true/false,
  "confidence": 0.0-1.0,
  "reason": "핵심 판단 이유",
  "issues": ["이슈1", "이슈2"],
  "claim_checks": [
    {{"claim_text": "...", "ok": true/false, "severity": "fatal/warning", "reason": "..."}}
  ]
}}
""".strip()

@retry_with_exponential_backoff
def validate_answer(
    qa_type: str,
    question: str,
    answer: str,
    claims: List[Dict[str, Any]],
    struct_summary: Dict[str, Any],
    evidence_items: List[Dict[str, str]],
) -> Dict[str, Any]:
    """LLM 기반 답변 검증"""
    
    prompt = build_validation_prompt(
        qa_type=qa_type,
        question=question,
        answer=answer,
        claims=claims,
        struct_summary=struct_summary,
        evidence_items=evidence_items,
    )
    
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": VAL_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_completion_tokens=650,  # max_tokens 대신 max_completion_tokens 사용
        response_format={"type": "json_object"},
    )
    
    try:
        out = json.loads(resp.choices[0].message.content)
    except Exception:
        return {
            "is_valid": False, 
            "reason": "validation_json_parse_failed", 
            "confidence": 0.0, 
            "issues": ["validation_json_parse_failed"]
        }
    
    # Fatal claim이 있으면 무조건 invalid
    claim_checks = out.get("claim_checks", []) or []
    has_fatal = any(
        c.get("ok") is False and c.get("severity") == "fatal" 
        for c in claim_checks
    )
    
    is_valid = bool(out.get("is_valid", False)) and not has_fatal
    confidence = float(out.get("confidence", 0.0) or 0.0)
    reason = _safe_str(out.get("reason", ""))
    issues = out.get("issues", []) or []
    
    # 신뢰도가 너무 낮으면 invalid
    if is_valid and confidence < 0.55:
        return {
            "is_valid": False, 
            "reason": f"low_confidence:{confidence}", 
            "confidence": confidence, 
            "issues": issues + ["low_confidence"]
        }
    
    if has_fatal:
        issues.append("has_fatal_claim")
    
    return {
        "is_valid": is_valid, 
        "reason": reason, 
        "confidence": confidence, 
        "issues": issues
    }

# 테스트
print("=== LLM 기반 자연어 표현 시스템 테스트 (수정된 버전) ===")

# Multi-hop 예제 테스트
if all_questions.get('multi_hop'):
    test_q = all_questions['multi_hop'][0]
    print(f"\n[Multi-hop 테스트]")
    print(f"질문: {test_q['question'][:100]}...")
    
    # Answer struct 계산
    pattern_data = {
        'start_node': '토마토',
        'end_node': 'CROP',
        'hop_count': 2
    }
    
    answer_struct = compute_multihop_answer_struct(test_q['question'], pattern_data)
    
    # Evidence 준비 (수정된 부분)
    test_evidence = {}
    for tu_id in answer_struct.get('evidence_text_unit_ids', [])[:3]:
        tu_str = str(tu_id)
        if tu_str in text_unit_lookup:
            # text_unit_lookup의 값이 dict인지 str인지 확인
            lookup_value = text_unit_lookup[tu_str]
            if isinstance(lookup_value, dict):
                test_evidence[tu_str] = lookup_value.get('text', '')[:200]
            else:
                # 문자열인 경우 그대로 사용
                test_evidence[tu_str] = str(lookup_value)[:200]
    
    print(f"\nAnswer struct 요약:")
    print(f"- 경로 존재: {answer_struct.get('path_exists', False)}")
    print(f"- 경로: {' → '.join(answer_struct.get('path', []))}")
    print(f"- Evidence 준비됨: {len(test_evidence)}개")
    
    if answer_struct.get('path_exists'):
        result = generate_natural_language_answer(
            'multi_hop',
            test_q['question'],
            answer_struct,
            test_evidence
        )
        
        print(f"\n생성 결과:")
        print(f"- 유효성: {result['is_valid']}")
        print(f"- 신뢰도: {result['confidence']:.2f}")
        if result['is_valid']:
            print(f"- 답변: {result['answer'][:150]}...")
            print(f"- Claims 수: {len(result['claims'])}")
        else:
            print(f"- 실패 이유: {result['validation_reason']}")
            print(f"- 이슈: {result['issues']}")

## Cell 43: 전체 GT 생성 배치 처리

모든 QA 타입에 대해 Ground Truth를 생성하고, 유효하지 않은 샘플을 필터링합니다.

In [ ]:
# ==========================================
# Cell 43: 전체 GT 생성 배치 처리 (완전 개선 버전)
# ==========================================

from tqdm import tqdm
import os
import json
import time
from datetime import datetime
from collections import defaultdict
import re
import logging

# GT 생성 함수 매핑
answer_struct_functions = {
    'multi_hop': compute_multihop_answer_struct,
    'community_synthesis': compute_community_synthesis_answer_struct,
    'cross_context': compute_cross_context_answer_struct,
    'implicit_relationships': compute_implicit_relationships_answer_struct,
    'causal_chains': compute_causal_chains_answer_struct
}

def load_all_patterns(output_path):
    """모든 QA 타입의 패턴 파일을 로드하여 pattern_id로 매핑"""
    patterns_cache = {}
    
    # Multi-hop 패턴 로드
    multi_hop_files = [
        output_path / 'multi_hop' / '2hop_patterns.json',
        output_path / 'multi_hop' / '3hop_patterns.json'
    ]
    
    multi_hop_patterns = []
    for pattern_file in multi_hop_files:
        if pattern_file.exists():
            with open(pattern_file, 'r', encoding='utf-8') as f:
                file_patterns = json.load(f)
                multi_hop_patterns.extend(file_patterns)
    
    # pattern_id 매핑 생성 (multi_hop_0, multi_hop_1, ...)
    for i, pattern in enumerate(multi_hop_patterns):
        patterns_cache[f'multi_hop_{i}'] = pattern
    
    print(f"패턴 로드 완료: multi_hop {len(multi_hop_patterns)}개")
    
    # 향후 확장: 다른 QA 타입들의 패턴 파일도 로드 가능
    # patterns_cache['community_synthesis_X'] = ...
    # patterns_cache['causal_chains_X'] = ...
    
    return patterns_cache

def extract_entities_improved(qa_type: str, question_text: str, question_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    개선된 엔티티 추출 함수 - 모든 QA 타입에 대해 robust 추출
    """
    pattern_data = {}
    
    if qa_type == 'cross_context':
        # Cross-context 엔티티 추출 - 개선된 버전
        entity = question_data.get('entity', '')
        if not entity:
            # 패턴 1: "다중 문서에 걸친 [ENTITY] 정보를 통합하세요" 
            match = re.search(r'다중 문서에 걸친\s+(.+?)\s+정보를 통합', question_text)
            if match:
                entity = match.group(1).strip()
            else:
                # 패턴 2: "[ENTITY]에 대해 최소 3개의 서로 다른 문맥에서"
                match = re.search(r'^(.+?)에 대해 최소 \d+개의 서로 다른 문맥', question_text)
                if match:
                    entity = match.group(1).strip()
                else:
                    # 패턴 3: "[ENTITY]과(와) 관련된 정보를 분석하세요"
                    match = re.search(r'^(.+?)(?:와|과)\s*관련된 정보를 분석', question_text)
                    if match:
                        entity = match.group(1).strip()
                    else:
                        # 패턴 4: "다양한 문서에서 언급된 [ENTITY]의 정보를"
                        match = re.search(r'다양한 문서에서 언급된\s+(.+?)(?:의|이|가)\s*정보를', question_text)
                        if match:
                            entity = match.group(1).strip()
                        else:
                            # 패턴 5: "[ENTITY]이(가) N개 문맥에서 언급된"
                            match = re.search(r'^(.+?)(?:이|가)\s*\d+개 문맥에서 언급', question_text)
                            if match:
                                entity = match.group(1).strip()
                            else:
                                # 패턴 6: 대문자 연속 단어 찾기 (가장 긴 매치 선택)
                                matches = re.findall(r'\b([A-Z]+(?:\s+[A-Z]+)*)\b', question_text)
                                if matches:
                                    entity = max(matches, key=len)
                                else:
                                    # 패턴 7: 한글 엔티티
                                    common_entities = ['고추', '토마토', '온실', '파프리카', '양액', '병해충', '모종']
                                    for ent in common_entities:
                                        if ent in question_text:
                                            entity = ent
                                            break
                                    if not entity:
                                        entity = '고추'  # 기본값
            
            # 그래프에 실제로 존재하는지 확인하고 수정
            if entity and entity not in G:
                # 가능한 변형들 시도
                candidates = [
                    entity.replace('다중 문서에 걸친 ', ''),
                    entity.replace(' 정보', ''),
                    entity.split()[0] if ' ' in entity else entity,
                    entity.split()[-1] if ' ' in entity else entity
                ]
                
                for candidate in candidates:
                    if candidate and candidate in G:
                        entity = candidate
                        break
                else:
                    # 여전히 못찾으면 유사한 노드 찾기
                    best_match = None
                    for node in G.nodes():
                        if entity and (entity.lower() in node.lower() or node.lower() in entity.lower()):
                            best_match = node
                            break
                    if best_match:
                        entity = best_match
        
        pattern_data = {'entity': entity}
        
    elif qa_type == 'implicit_relationships':
        # Implicit relationships 두 엔티티 추출 - 개선된 버전
        entity1 = question_data.get('entity1', '')
        entity2 = question_data.get('entity2', '')
        
        if not entity1 or not entity2:
            # 패턴 1: "직접 연결되지 않은 [ENTITY1]과(와) [ENTITY2] 사이의"
            match = re.search(r'직접 연결되지 않은\s+(.+?)(?:과|와)\s+(.+?)\s+사이의', question_text)
            if match:
                entity1 = match.group(1).strip()
                entity2 = match.group(2).strip()
            else:
                # 패턴 2: "[ENTITY1]과(와) [ENTITY2]의 숨겨진"
                match = re.search(r'^(.+?)(?:과|와)\s+(.+?)의 숨겨진', question_text)
                if match:
                    entity1 = match.group(1).strip()
                    entity2 = match.group(2).strip()
                else:
                    # 패턴 3: "[ENTITY1]과(와) [ENTITY2] 사이의 간접"
                    match = re.search(r'^(.+?)(?:과|와)\s+(.+?)\s+사이의 간접', question_text)
                    if match:
                        entity1 = match.group(1).strip()
                        entity2 = match.group(2).strip()
                    else:
                        # 패턴 4: 가장 관대한 패턴 - "과(와)" 기준 분할
                        if "과" in question_text or "와" in question_text:
                            parts = re.split(r'\s*(?:과|와)\s*', question_text, maxsplit=1)
                            if len(parts) == 2:
                                # 첫 번째 부분에서 entity1 추출
                                part1_words = parts[0].split()
                                if part1_words:
                                    entity1 = ' '.join(part1_words[-3:]).strip()
                                    entity1 = re.sub(r'^.*?((?:[A-Z\s]+|[가-힣]+(?:\s+[가-힣]+)*))$', r'\1', entity1)
                                
                                # 두 번째 부분에서 entity2 추출  
                                part2_words = parts[1].split()
                                if part2_words:
                                    entity2 = ' '.join(part2_words[:3]).strip()
                                    entity2 = re.sub(r'^((?:[A-Z\s]+|[가-힣]+(?:\s+[가-힣]+)*)).*$', r'\1', entity2)
                        
                        if not entity1 or not entity2:
                            entity1 = '고추'
                            entity2 = '토마토'
            
            # 추출된 엔티티 정리
            if entity1:
                entity1 = entity1.replace('직접 연결되지 않은', '').strip()
            if entity2:
                entity2 = re.sub(r'\s*(사이의|간의|의).*$', '', entity2).strip()
            
            # 그래프에 실제로 존재하는지 확인하고 유사 노드로 대체
            for i, entity in enumerate([entity1, entity2], 1):
                if entity and entity not in G:
                    # 유사한 노드 찾기
                    best_match = None
                    for node in G.nodes():
                        if entity.lower() in node.lower() or node.lower() in entity.lower():
                            best_match = node
                            break
                    
                    if best_match:
                        if i == 1:
                            entity1 = best_match
                        else:
                            entity2 = best_match
        
        pattern_data = {'entity1': entity1, 'entity2': entity2}
        
    elif qa_type == 'community_synthesis':
        pattern_data = {
            'community_id': question_data.get('community_id', 1),
            'level': question_data.get('level', 0)
        }
        
    elif qa_type == 'causal_chains':
        pattern_data = {
            'chain_type': question_data.get('chain_type', 'linear'),
            'start_event': question_data.get('start_event', '고추'),
            'terminal_effects': question_data.get('terminal_effects', [])
        }
        
    else:  # multi_hop은 별도 처리
        pattern_data = {}
    
    return pattern_data

def process_single_question(qa_type: str, question_data: Dict[str, Any], 
                           evidence_texts: Dict[str, str], 
                           patterns_cache: Dict[str, Any] = None) -> Dict[str, Any]:
    """
    단일 질문에 대한 GT 생성 처리 (완전 개선 버전)
    
    Returns:
        결과 딕셔너리 (질문, 답변, 유효성 등 포함)
    """
    question_text = question_data['question']
    pattern_id = question_data.get('pattern_id', '')
    
    # Pattern data 추출
    if qa_type == 'multi_hop':
        # Multi-hop는 패턴 캐시 우선 사용
        if patterns_cache and pattern_id in patterns_cache:
            pattern = patterns_cache[pattern_id]
            path = pattern.get('path', [])
            
            pattern_data = {
                'start_node': path[0] if len(path) > 0 else '',
                'end_node': path[-1] if len(path) > 1 else '',
                'hop_count': pattern.get('hop_count', len(path) - 1 if path else 2),
                'path': path,
                'entities': pattern.get('entities', []),
                'relationships': pattern.get('relationships', [])
            }
        else:
            # 패턴이 없으면 질문에서 추출 시도
            extracted = extract_nodes_from_multihop_question(question_text)
            if extracted:
                pattern_data = extracted
            else:
                pattern_data = {'start_node': '', 'end_node': '', 'hop_count': 2}
    else:
        # 다른 QA 타입들은 개선된 엔티티 추출 사용
        pattern_data = extract_entities_improved(qa_type, question_text, question_data)
    
    try:
        # 1. Answer struct 계산
        answer_struct_func = answer_struct_functions[qa_type]
        answer_struct = answer_struct_func(question_text, pattern_data)
        
        # 2. 자연어 답변 생성 및 검증
        result = generate_natural_language_answer(
            qa_type=qa_type,
            question=question_text,
            answer_struct=answer_struct,
            evidence_texts=evidence_texts
        )
        
        # 3. 결과 구성
        return {
            'qa_type': qa_type,
            'question_id': question_data.get('question_id', ''),
            'pattern_id': pattern_id,
            'question': question_text,
            'pattern_data': pattern_data,
            'answer_struct': answer_struct,
            'answer': result.get('answer', ''),
            'claims': result.get('claims', []),
            'is_valid': result.get('is_valid', False),
            'validation_reason': result.get('validation_reason', ''),
            'confidence': result.get('confidence', 0.0),
            'issues': result.get('issues', []),
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        logging.error(f"Error processing question: {str(e)}")
        return {
            'qa_type': qa_type,
            'question_id': question_data.get('question_id', ''),
            'pattern_id': pattern_id,
            'question': question_text,
            'pattern_data': pattern_data,
            'answer': '',
            'is_valid': False,
            'validation_reason': f'processing_error: {str(e)}',
            'confidence': 0.0,
            'timestamp': datetime.now().isoformat()
        }

def batch_generate_ground_truths(
    all_questions: Dict[str, List[Dict]], 
    evidence_texts: Dict[str, str],
    batch_size: int = 10,
    output_dir: str = None
) -> Dict[str, List[Dict]]:
    """
    모든 질문에 대해 배치로 GT를 생성 (완전 개선 버전)
    
    Args:
        all_questions: QA 타입별 질문 딕셔너리
        evidence_texts: text_unit_id -> text 매핑
        batch_size: 배치 크기
        output_dir: 결과 저장 디렉토리
        
    Returns:
        QA 타입별 GT 결과 딕셔너리
    """
    
    # 패턴 파일들 로드
    patterns_cache = load_all_patterns(output_path)
    
    results = defaultdict(list)
    stats = defaultdict(lambda: {'total': 0, 'valid': 0, 'invalid': 0})
    
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    # 전체 질문 수 계산
    total_questions = sum(len(questions) for questions in all_questions.values())
    
    print(f"\n{'='*80}")
    print(f"GT 생성 시작 (완전 개선 버전)")
    print(f"{'='*80}")
    print(f"전체 질문 수: {total_questions}")
    print(f"배치 크기: {batch_size}")
    print(f"로드된 패턴 수: {len(patterns_cache)}")
    print(f"QA 타입: {list(all_questions.keys())}")
    
    # 전체 진행률 표시용 tqdm
    with tqdm(total=total_questions, desc="전체 GT 생성") as pbar:
        
        for qa_type, questions in all_questions.items():
            print(f"\n\n{'='*60}")
            print(f"{qa_type.upper()} 처리 시작 ({len(questions)}개)")
            print(f"{'='*60}")
            
            # 처음 몇 개 질문의 패턴 매핑 확인
            if qa_type == 'multi_hop' and len(questions) > 0:
                test_q = questions[0]
                pattern_id = test_q.get('pattern_id', '')
                if pattern_id in patterns_cache:
                    pattern = patterns_cache[pattern_id]
                    print(f"✓ 패턴 매핑 확인: {pattern_id} -> {pattern.get('path', [])}")
                else:
                    print(f"⚠️ 패턴 매핑 실패: {pattern_id}")
            
            # 배치 처리
            for i in range(0, len(questions), batch_size):
                batch = questions[i:i+batch_size]
                batch_results = []
                
                # 배치 내 개별 처리
                for j, question_data in enumerate(batch):
                    stats[qa_type]['total'] += 1
                    
                    # GT 생성 (패턴 캐시 전달)
                    result = process_single_question(
                        qa_type, question_data, evidence_texts, patterns_cache
                    )
                    
                    if result['is_valid']:
                        stats[qa_type]['valid'] += 1
                        status = "✓"
                    else:
                        stats[qa_type]['invalid'] += 1
                        status = "✗"
                    
                    batch_results.append(result)
                    
                    # 진행 상황 업데이트
                    pbar.update(1)
                    pbar.set_description(
                        f"{qa_type}: {i+j+1}/{len(questions)} "
                        f"(유효: {stats[qa_type]['valid']}, "
                        f"무효: {stats[qa_type]['invalid']})"
                    )
                    
                    # 개별 결과 출력 (선택적)
                    if j < 2:  # 각 배치의 처음 2개만 샘플 출력
                        print(f"\n[{status}] 질문: {result['question'][:80]}...")
                        if result['is_valid']:
                            print(f"    답변: {result['answer'][:120]}...")
                        else:
                            print(f"    실패: {result['validation_reason']}")
                    
                    # API 속도 제한 대응
                    time.sleep(0.5)
                
                results[qa_type].extend(batch_results)
                
                # 중간 저장 (배치마다)
                if output_dir and (i + batch_size) % (batch_size * 5) == 0:
                    save_intermediate_results(results, stats, output_dir, qa_type)
            
            # QA 타입별 통계 출력
            print(f"\n{qa_type} 완료:")
            print(f"  - 전체: {stats[qa_type]['total']}")
            print(f"  - 유효: {stats[qa_type]['valid']} ({stats[qa_type]['valid']/stats[qa_type]['total']*100:.1f}%)")
            print(f"  - 무효: {stats[qa_type]['invalid']} ({stats[qa_type]['invalid']/stats[qa_type]['total']*100:.1f}%)")
    
    # 최종 통계
    print(f"\n\n{'='*80}")
    print(f"GT 생성 완료 - 최종 통계")
    print(f"{'='*80}")
    
    total_valid = sum(s['valid'] for s in stats.values())
    total_invalid = sum(s['invalid'] for s in stats.values())
    
    print(f"전체 질문: {total_questions}")
    print(f"유효 GT: {total_valid} ({total_valid/total_questions*100:.1f}%)")
    print(f"무효 GT: {total_invalid} ({total_invalid/total_questions*100:.1f}%)")
    
    print(f"\nQA 타입별 유효율:")
    for qa_type in stats:
        valid_rate = stats[qa_type]['valid'] / stats[qa_type]['total'] * 100
        print(f"  - {qa_type}: {valid_rate:.1f}%")
    
    # 최종 저장
    if output_dir:
        save_final_results(results, stats, output_dir)
    
    return dict(results)

def save_intermediate_results(results: Dict[str, List], stats: Dict, 
                            output_dir: str, current_type: str):
    """중간 결과 저장"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if current_type in results:
        filename = os.path.join(output_dir, f"intermediate_{current_type}_{timestamp}.json")
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump({
                'qa_type': current_type,
                'results': results[current_type],
                'stats': dict(stats[current_type]),
                'timestamp': timestamp
            }, f, ensure_ascii=False, indent=2)
        print(f"\n중간 저장: {filename}")

def save_final_results(results: Dict[str, List], stats: Dict, output_dir: str):
    """최종 결과 저장"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. 전체 결과 저장
    all_results_file = os.path.join(output_dir, f"all_gt_results_{timestamp}.json")
    with open(all_results_file, 'w', encoding='utf-8') as f:
        json.dump({
            'results': dict(results),
            'stats': {k: dict(v) for k, v in stats.items()},
            'timestamp': timestamp,
            'summary': {
                'total_questions': sum(s['total'] for s in stats.values()),
                'total_valid': sum(s['valid'] for s in stats.values()),
                'total_invalid': sum(s['invalid'] for s in stats.values()),
                'overall_valid_rate': sum(s['valid'] for s in stats.values()) / sum(s['total'] for s in stats.values())
            }
        }, f, ensure_ascii=False, indent=2)
    
    # 2. QA 타입별 유효한 결과만 저장
    for qa_type, type_results in results.items():
        valid_results = [r for r in type_results if r['is_valid']]
        
        if valid_results:
            valid_file = os.path.join(output_dir, f"valid_gt_{qa_type}_{timestamp}.json")
            with open(valid_file, 'w', encoding='utf-8') as f:
                json.dump(valid_results, f, ensure_ascii=False, indent=2)
            
            print(f"\n{qa_type} 유효 GT 저장: {valid_file} ({len(valid_results)}개)")
    
    # 3. 통계 요약 저장
    stats_file = os.path.join(output_dir, f"gt_generation_stats_{timestamp}.json")
    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump({
            'stats': {k: dict(v) for k, v in stats.items()},
            'summary': {
                'total_questions': sum(s['total'] for s in stats.values()),
                'total_valid': sum(s['valid'] for s in stats.values()),
                'total_invalid': sum(s['invalid'] for s in stats.values()),
                'overall_valid_rate': sum(s['valid'] for s in stats.values()) / sum(s['total'] for s in stats.values())
            },
            'timestamp': timestamp
        }, f, ensure_ascii=False, indent=2)
    
    print(f"\n최종 결과 저장 완료:")
    print(f"  - 전체 결과: {all_results_file}")
    print(f"  - 통계 요약: {stats_file}")

# ===========================================
# 테스트 및 실행 준비
# ===========================================

print("=" * 80)
print("GT 생성 배치 처리 시스템 (완전 개선 버전) 준비 완료")
print("=" * 80)

print(f"\n주요 개선사항:")
print(f"  ✅ Multi-hop: 패턴 파일에서 start_node, end_node, path 정보 추출")
print(f"  ✅ Cross-context: 개선된 엔티티 추출 (7단계 패턴 + 유사 노드 대체)")
print(f"  ✅ Implicit relationships: 4단계 패턴 + 자동 엔티티 정리 + 유사 노드 대체")
print(f"  ✅ Causal chains: 이미 완벽하게 작동 중")
print(f"  ✅ Community synthesis: 기존 로직 유지")
print(f"  ✅ 통합 error handling 및 상세한 진행 상황 표시")

print(f"\n실행 명령:")
print(f"gt_results = batch_generate_ground_truths(")
print(f"    all_questions=all_questions,")
print(f"    evidence_texts=text_unit_lookup,")
print(f"    batch_size=5,")  
print(f"    output_dir=str(output_path / 'generated_ground_truths')")
print(f")")

# 빠른 테스트용 - 각 QA 타입별 1개 질문만
print(f"\n빠른 테스트용 (각 타입 1개씩):")
print(f"quick_questions = {{k: v[:1] for k, v in all_questions.items()}}")
print(f"quick_results = batch_generate_ground_truths(quick_questions, text_unit_lookup, batch_size=1)")

print(f"\n=" * 80)

In [ ]:
gt_results = batch_generate_ground_truths(
    all_questions=all_questions,
    evidence_texts=text_unit_lookup,
    batch_size=10,
    output_dir=str(output_path / 'generated_ground_truths')
)

## Cell 44: 최종 데이터셋 구성 및 검증

생성된 GT 결과를 통합하여 최종 QA 데이터셋을 구성하고 품질을 검증합니다.

In [ ]:
# 최종 데이터셋 구성 및 검증
import pandas as pd
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

def analyze_gt_results(gt_results: Dict[str, List[Dict]]) -> Dict[str, Any]:
    """
    GT 생성 결과를 분석하여 데이터셋 품질 메트릭을 계산
    
    Args:
        gt_results: QA 타입별 GT 결과
        
    Returns:
        분석 결과 딕셔너리
    """
    analysis = {
        'summary': {},
        'quality_metrics': {},
        'qa_type_stats': {},
        'validation_issues': defaultdict(Counter),
        'confidence_distribution': {},
        'dataset_recommendations': []
    }
    
    total_questions = 0
    total_valid = 0
    total_invalid = 0
    
    # QA 타입별 통계
    for qa_type, results in gt_results.items():
        valid_results = [r for r in results if r.get('is_valid', False)]
        invalid_results = [r for r in results if not r.get('is_valid', False)]
        
        type_stats = {
            'total': len(results),
            'valid': len(valid_results),
            'invalid': len(invalid_results),
            'valid_rate': len(valid_results) / len(results) if results else 0,
            'avg_confidence': sum(r.get('confidence', 0) for r in valid_results) / len(valid_results) if valid_results else 0,
            'avg_answer_length': sum(len(r.get('answer', '')) for r in valid_results) / len(valid_results) if valid_results else 0,
            'avg_claims_count': sum(len(r.get('claims', [])) for r in valid_results) / len(valid_results) if valid_results else 0
        }
        
        analysis['qa_type_stats'][qa_type] = type_stats
        
        total_questions += len(results)
        total_valid += len(valid_results)
        total_invalid += len(invalid_results)
        
        # 검증 실패 이유 분석
        for result in invalid_results:
            reason = result.get('validation_reason', 'unknown')
            issues = result.get('issues', [])
            
            analysis['validation_issues'][qa_type][reason] += 1
            for issue in issues:
                analysis['validation_issues'][qa_type][f"issue:{issue}"] += 1
        
        # 신뢰도 분포
        confidences = [r.get('confidence', 0) for r in valid_results]
        if confidences:
            analysis['confidence_distribution'][qa_type] = {
                'mean': sum(confidences) / len(confidences),
                'min': min(confidences),
                'max': max(confidences),
                'std': (sum((x - sum(confidences)/len(confidences))**2 for x in confidences) / len(confidences))**0.5,
                'bins': {
                    '0.0-0.5': len([c for c in confidences if 0.0 <= c < 0.5]),
                    '0.5-0.7': len([c for c in confidences if 0.5 <= c < 0.7]),
                    '0.7-0.9': len([c for c in confidences if 0.7 <= c < 0.9]),
                    '0.9-1.0': len([c for c in confidences if 0.9 <= c <= 1.0])
                }
            }
    
    # 전체 요약
    analysis['summary'] = {
        'total_questions': total_questions,
        'total_valid': total_valid,
        'total_invalid': total_invalid,
        'overall_valid_rate': total_valid / total_questions if total_questions > 0 else 0,
        'target_dataset_size': total_valid,
        'quality_score': calculate_quality_score(analysis['qa_type_stats'])
    }
    
    # 품질 메트릭
    analysis['quality_metrics'] = {
        'balance_score': calculate_balance_score(analysis['qa_type_stats']),
        'confidence_score': calculate_confidence_score(analysis['confidence_distribution']),
        'completeness_score': calculate_completeness_score(analysis['qa_type_stats']),
        'consistency_score': calculate_consistency_score(gt_results)
    }
    
    # 데이터셋 권장사항
    analysis['dataset_recommendations'] = generate_dataset_recommendations(analysis)
    
    return analysis

def calculate_quality_score(qa_type_stats: Dict) -> float:
    """전체 데이터셋 품질 점수 계산 (0-1)"""
    if not qa_type_stats:
        return 0.0
    
    # 각 QA 타입의 품질 점수
    type_scores = []
    for qa_type, stats in qa_type_stats.items():
        valid_rate = stats['valid_rate']
        avg_confidence = stats['avg_confidence']
        
        # 최소 유효 샘플 수 요구
        min_samples_score = min(stats['valid'] / 20, 1.0)  # 20개 이상이면 만점
        
        type_score = (valid_rate * 0.4 + avg_confidence * 0.4 + min_samples_score * 0.2)
        type_scores.append(type_score)
    
    return sum(type_scores) / len(type_scores)

def calculate_balance_score(qa_type_stats: Dict) -> float:
    """QA 타입 간 균형 점수 계산"""
    if not qa_type_stats:
        return 0.0
    
    valid_counts = [stats['valid'] for stats in qa_type_stats.values()]
    if not valid_counts:
        return 0.0
    
    mean_count = sum(valid_counts) / len(valid_counts)
    if mean_count == 0:
        return 0.0
    
    # 표준편차 기반 균형 점수 (낮을수록 좋음)
    variance = sum((count - mean_count)**2 for count in valid_counts) / len(valid_counts)
    std_dev = variance**0.5
    
    # 정규화: 표준편차가 평균의 50% 이하면 만점
    balance_score = max(0, 1 - (std_dev / (mean_count * 0.5)))
    return balance_score

def calculate_confidence_score(confidence_dist: Dict) -> float:
    """신뢰도 분포 점수 계산"""
    if not confidence_dist:
        return 0.0
    
    total_score = 0
    total_types = 0
    
    for qa_type, dist in confidence_dist.items():
        # 평균 신뢰도가 0.7 이상이고, 0.9+ 샘플이 충분하면 좋은 점수
        mean_conf = dist['mean']
        high_conf_ratio = (dist['bins']['0.7-0.9'] + dist['bins']['0.9-1.0']) / sum(dist['bins'].values())
        
        type_score = (mean_conf * 0.6 + high_conf_ratio * 0.4)
        total_score += type_score
        total_types += 1
    
    return total_score / total_types if total_types > 0 else 0

def calculate_completeness_score(qa_type_stats: Dict) -> float:
    """데이터셋 완성도 점수 계산"""
    expected_types = {'multi_hop', 'community_synthesis', 'cross_context', 'implicit_relationships', 'causal_chains'}
    
    if not qa_type_stats:
        return 0.0
    
    # 모든 QA 타입이 있는지 확인
    type_coverage = len(set(qa_type_stats.keys()) & expected_types) / len(expected_types)
    
    # 각 타입의 최소 샘플 수 확인
    min_samples_coverage = 0
    for qa_type in expected_types:
        if qa_type in qa_type_stats:
            if qa_type_stats[qa_type]['valid'] >= 10:  # 최소 10개
                min_samples_coverage += 1
    min_samples_coverage /= len(expected_types)
    
    return (type_coverage * 0.5 + min_samples_coverage * 0.5)

def calculate_consistency_score(gt_results: Dict) -> float:
    """데이터셋 일관성 점수 계산"""
    # 간단한 휴리스틱: 답변 길이와 Claims 수의 일관성
    total_score = 0
    total_types = 0
    
    for qa_type, results in gt_results.items():
        valid_results = [r for r in results if r.get('is_valid', False)]
        if len(valid_results) < 3:
            continue
            
        answer_lengths = [len(r.get('answer', '')) for r in valid_results]
        claims_counts = [len(r.get('claims', [])) for r in valid_results]
        
        if not answer_lengths or not claims_counts:
            continue
        
        # 변동계수 계산 (표준편차/평균) - 낮을수록 일관성 높음
        def cv(values):
            mean_val = sum(values) / len(values)
            if mean_val == 0:
                return 1.0
            variance = sum((x - mean_val)**2 for x in values) / len(values)
            return (variance**0.5) / mean_val
        
        length_cv = cv(answer_lengths)
        claims_cv = cv(claims_counts)
        
        # CV가 0.5 이하면 일관성 있음
        consistency = 1 - min((length_cv + claims_cv) / 1.0, 1.0)
        total_score += consistency
        total_types += 1
    
    return total_score / total_types if total_types > 0 else 0

def generate_dataset_recommendations(analysis: Dict) -> List[str]:
    """데이터셋 개선 권장사항 생성"""
    recommendations = []
    
    summary = analysis['summary']
    qa_stats = analysis['qa_type_stats']
    quality_metrics = analysis['quality_metrics']
    
    # 전체 유효율 체크
    if summary['overall_valid_rate'] < 0.5:
        recommendations.append(
            f"⚠️ 전체 유효율이 {summary['overall_valid_rate']:.1%}로 낮습니다. "
            "질문 생성 알고리즘이나 검증 기준을 재검토하세요."
        )
    elif summary['overall_valid_rate'] < 0.7:
        recommendations.append(
            f"📋 전체 유효율이 {summary['overall_valid_rate']:.1%}입니다. "
            "타겟 70% 이상을 위해 개선이 필요합니다."
        )
    
    # QA 타입별 권장사항
    for qa_type, stats in qa_stats.items():
        if stats['valid'] < 10:
            recommendations.append(
                f"📊 {qa_type}: 유효 샘플이 {stats['valid']}개뿐입니다. "
                "최소 10개 이상 확보를 권장합니다."
            )
        elif stats['valid_rate'] < 0.5:
            recommendations.append(
                f"🔧 {qa_type}: 유효율이 {stats['valid_rate']:.1%}로 낮습니다. "
                "패턴 생성이나 검증 로직을 개선하세요."
            )
    
    # 균형 체크
    if quality_metrics['balance_score'] < 0.6:
        valid_counts = {qa_type: stats['valid'] for qa_type, stats in qa_stats.items()}
        max_type = max(valid_counts, key=valid_counts.get)
        min_type = min(valid_counts, key=valid_counts.get)
        recommendations.append(
            f"⚖️ QA 타입 간 불균형이 있습니다. "
            f"{max_type}({valid_counts[max_type]}개) vs {min_type}({valid_counts[min_type]}개). "
            "균형있는 샘플링을 고려하세요."
        )
    
    # 신뢰도 체크
    if quality_metrics['confidence_score'] < 0.7:
        recommendations.append(
            "🎯 신뢰도가 전반적으로 낮습니다. "
            "Evidence 품질을 개선하거나 생성 프롬프트를 최적화하세요."
        )
    
    # 긍정적 피드백
    if quality_metrics['confidence_score'] > 0.8 and summary['overall_valid_rate'] > 0.7:
        recommendations.append(
            "✅ 데이터셋 품질이 우수합니다! "
            f"유효율 {summary['overall_valid_rate']:.1%}, 신뢰도 점수 {quality_metrics['confidence_score']:.2f}"
        )
    
    if not recommendations:
        recommendations.append("✨ 특별한 개선사항이 발견되지 않았습니다. 데이터셋이 양호합니다.")
    
    return recommendations

def create_final_dataset(
    gt_results: Dict[str, List[Dict]],
    output_path: Path,
    min_confidence: float = 0.6,
    balance_sampling: bool = True,
    max_samples_per_type: int = None
) -> Dict[str, Any]:
    """
    최종 데이터셋 생성 및 저장
    
    Args:
        gt_results: GT 생성 결과
        output_path: 저장 경로
        min_confidence: 최소 신뢰도 기준
        balance_sampling: 균형 샘플링 여부
        max_samples_per_type: QA 타입별 최대 샘플 수
        
    Returns:
        최종 데이터셋 정보
    """
    print(f"=== 최종 데이터셋 생성 ===")
    print(f"최소 신뢰도 기준: {min_confidence}")
    print(f"균형 샘플링: {balance_sampling}")
    print(f"타입별 최대 샘플: {max_samples_per_type or '제한없음'}")
    
    final_dataset = []
    dataset_stats = defaultdict(int)
    
    # 1. 필터링 및 선별
    filtered_results = {}
    for qa_type, results in gt_results.items():
        # 유효하고 신뢰도가 기준 이상인 결과만 선택
        valid_results = [
            r for r in results
            if r.get('is_valid', False) and r.get('confidence', 0) >= min_confidence
        ]
        
        # 신뢰도 순으로 정렬
        valid_results.sort(key=lambda x: x.get('confidence', 0), reverse=True)
        
        # 최대 샘플 수 제한
        if max_samples_per_type and len(valid_results) > max_samples_per_type:
            valid_results = valid_results[:max_samples_per_type]
        
        filtered_results[qa_type] = valid_results
        print(f"  {qa_type}: {len(valid_results)}개 선별")
    
    # 2. 균형 샘플링 (선택사항)
    if balance_sampling:
        # 가장 적은 타입의 샘플 수에 맞춤
        min_samples = min(len(results) for results in filtered_results.values())
        if min_samples > 0:
            print(f"\n균형 샘플링: 타입별 {min_samples}개로 조정")
            for qa_type in filtered_results:
                if len(filtered_results[qa_type]) > min_samples:
                    # 균등하게 분산된 샘플 선택
                    step = len(filtered_results[qa_type]) / min_samples
                    indices = [int(i * step) for i in range(min_samples)]
                    filtered_results[qa_type] = [filtered_results[qa_type][i] for i in indices]
    
    # 3. 최종 데이터셋 구성
    dataset_id = 0
    for qa_type, results in filtered_results.items():
        for result in results:
            final_sample = {
                'id': f"qa_{dataset_id:04d}",
                'qa_type': qa_type,
                'question': result['question'],
                'answer': result['answer'],
                'claims': result['claims'],
                'confidence': result['confidence'],
                'pattern_data': result.get('pattern_data', {}),
                'answer_struct_summary': summarize_struct_for_validation(qa_type, result.get('answer_struct', {})),
                'evidence_count': len(result.get('answer_struct', {}).get('evidence_text_unit_ids', [])),
                'metadata': {
                    'generated_at': result.get('timestamp'),
                    'qa_type': qa_type,
                    'validation_passed': True,
                    'issues': result.get('issues', [])
                }
            }
            
            final_dataset.append(final_sample)
            dataset_stats[qa_type] += 1
            dataset_id += 1
    
    # 4. 저장
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # JSON 형식으로 저장
    dataset_file = output_path / f"korean_agriculture_qa_dataset_{timestamp}.json"
    with open(dataset_file, 'w', encoding='utf-8') as f:
        json.dump({
            'dataset_info': {
                'name': 'Korean Agriculture Knowledge Graph QA Dataset',
                'version': '1.0',
                'created_at': timestamp,
                'total_samples': len(final_dataset),
                'qa_types': dict(dataset_stats),
                'filtering_criteria': {
                    'min_confidence': min_confidence,
                    'balance_sampling': balance_sampling,
                    'max_samples_per_type': max_samples_per_type
                }
            },
            'samples': final_dataset
        }, f, ensure_ascii=False, indent=2)
    
    # CSV 형식으로도 저장 (분석 편의용)
    csv_data = []
    for sample in final_dataset:
        csv_row = {
            'id': sample['id'],
            'qa_type': sample['qa_type'],
            'question': sample['question'],
            'answer': sample['answer'],
            'confidence': sample['confidence'],
            'answer_length': len(sample['answer']),
            'claims_count': len(sample['claims']),
            'evidence_count': sample['evidence_count']
        }
        csv_data.append(csv_row)
    
    df = pd.DataFrame(csv_data)
    csv_file = output_path / f"korean_agriculture_qa_dataset_{timestamp}.csv"
    df.to_csv(csv_file, index=False, encoding='utf-8-sig')
    
    # 통계 정보 저장
    stats_info = {
        'dataset_summary': {
            'total_samples': len(final_dataset),
            'qa_type_distribution': dict(dataset_stats),
            'avg_confidence': sum(s['confidence'] for s in final_dataset) / len(final_dataset),
            'avg_answer_length': sum(len(s['answer']) for s in final_dataset) / len(final_dataset),
            'avg_claims_count': sum(len(s['claims']) for s in final_dataset) / len(final_dataset)
        },
        'quality_metrics': {
            'confidence_distribution': df['confidence'].describe().to_dict(),
            'answer_length_distribution': df['answer_length'].describe().to_dict(),
            'claims_count_distribution': df['claims_count'].describe().to_dict()
        },
        'files_created': {
            'dataset_json': str(dataset_file),
            'dataset_csv': str(csv_file)
        }
    }
    
    stats_file = output_path / f"dataset_statistics_{timestamp}.json"
    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump(stats_info, f, ensure_ascii=False, indent=2)
    
    print(f"\n=== 데이터셋 생성 완료 ===")
    print(f"최종 샘플 수: {len(final_dataset)}")
    print(f"QA 타입별 분포:")
    for qa_type, count in dataset_stats.items():
        print(f"  - {qa_type}: {count}개")
    print(f"\n저장된 파일:")
    print(f"  - JSON: {dataset_file}")
    print(f"  - CSV: {csv_file}")
    print(f"  - 통계: {stats_file}")
    
    return {
        'dataset': final_dataset,
        'statistics': stats_info,
        'files': {
            'json': dataset_file,
            'csv': csv_file,
            'stats': stats_file
        }
    }

def visualize_dataset_quality(analysis: Dict[str, Any], save_path: Path = None):
    """데이터셋 품질 시각화"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Korean Agriculture QA Dataset Quality Analysis', fontsize=16, fontweight='bold')
    
    # 1. QA 타입별 유효율
    qa_types = list(analysis['qa_type_stats'].keys())
    valid_rates = [analysis['qa_type_stats'][qt]['valid_rate'] for qt in qa_types]
    
    axes[0, 0].bar(qa_types, valid_rates, color='skyblue', alpha=0.7)
    axes[0, 0].set_title('Valid Rate by QA Type')
    axes[0, 0].set_ylabel('Valid Rate')
    axes[0, 0].set_ylim(0, 1)
    for i, v in enumerate(valid_rates):
        axes[0, 0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. 샘플 수 분포
    valid_counts = [analysis['qa_type_stats'][qt]['valid'] for qt in qa_types]
    
    axes[0, 1].bar(qa_types, valid_counts, color='lightgreen', alpha=0.7)
    axes[0, 1].set_title('Valid Sample Count by QA Type')
    axes[0, 1].set_ylabel('Sample Count')
    for i, v in enumerate(valid_counts):
        axes[0, 1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. 신뢰도 분포
    conf_means = []
    conf_stds = []
    for qt in qa_types:
        if qt in analysis['confidence_distribution']:
            conf_means.append(analysis['confidence_distribution'][qt]['mean'])
            conf_stds.append(analysis['confidence_distribution'][qt]['std'])
        else:
            conf_means.append(0)
            conf_stds.append(0)
    
    axes[0, 2].bar(qa_types, conf_means, yerr=conf_stds, color='orange', alpha=0.7, capsize=5)
    axes[0, 2].set_title('Confidence Score by QA Type')
    axes[0, 2].set_ylabel('Confidence')
    axes[0, 2].set_ylim(0, 1)
    for i, v in enumerate(conf_means):
        axes[0, 2].text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')
    axes[0, 2].tick_params(axis='x', rotation=45)
    
    # 4. 품질 메트릭 레이더 차트
    metrics = analysis['quality_metrics']
    metric_names = list(metrics.keys())
    metric_values = list(metrics.values())
    
    axes[1, 0].remove()  # 레이더 차트를 위해 제거
    ax_radar = fig.add_subplot(2, 3, 4, projection='polar')
    
    angles = [i * 2 * 3.14159 / len(metric_names) for i in range(len(metric_names))]
    angles += angles[:1]  # 원형 그래프를 위해 첫 번째 값 추가
    
    metric_values += metric_values[:1]
    ax_radar.plot(angles, metric_values, 'o-', linewidth=2, color='red', alpha=0.7)
    ax_radar.fill(angles, metric_values, alpha=0.25, color='red')
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels([name.replace('_', '\n') for name in metric_names])
    ax_radar.set_ylim(0, 1)
    ax_radar.set_title('Quality Metrics')
    
    # 5. 검증 실패 이유 분포
    all_issues = Counter()
    for qa_type, issues in analysis['validation_issues'].items():
        for issue, count in issues.items():
            all_issues[issue] += count
    
    if all_issues:
        top_issues = all_issues.most_common(8)
        issue_names = [name[:20] + '...' if len(name) > 20 else name for name, _ in top_issues]
        issue_counts = [count for _, count in top_issues]
        
        axes[1, 1].barh(issue_names, issue_counts, color='salmon', alpha=0.7)
        axes[1, 1].set_title('Top Validation Issues')
        axes[1, 1].set_xlabel('Count')
    else:
        axes[1, 1].text(0.5, 0.5, 'No Validation Issues', ha='center', va='center', transform=axes[1, 1].transAxes)
        axes[1, 1].set_title('Validation Issues')
    
    # 6. 전체 요약 텍스트
    axes[1, 2].axis('off')
    summary_text = f"""
Dataset Summary:
• Total Questions: {analysis['summary']['total_questions']:,}
• Valid GT: {analysis['summary']['total_valid']:,}
• Overall Valid Rate: {analysis['summary']['overall_valid_rate']:.1%}
• Quality Score: {analysis['summary']['quality_score']:.2f}/1.0

Quality Metrics:
• Balance: {metrics['balance_score']:.2f}
• Confidence: {metrics['confidence_score']:.2f}
• Completeness: {metrics['completeness_score']:.2f}
• Consistency: {metrics['consistency_score']:.2f}

Recommendations:
{chr(10).join(f'• {rec[:60]}...' if len(rec) > 60 else f'• {rec}' for rec in analysis['dataset_recommendations'][:4])}
    """.strip()
    
    axes[1, 2].text(0.05, 0.95, summary_text, transform=axes[1, 2].transAxes, 
                    fontsize=10, verticalalignment='top', fontfamily='monospace')
    axes[1, 2].set_title('Summary & Recommendations')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path / f'dataset_quality_analysis_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png', 
                   dpi=300, bbox_inches='tight')
        print(f"시각화 저장됨: {save_path}")
    
    plt.show()

# 시스템 준비 완료
print("=== 최종 데이터셋 구성 및 검증 시스템 준비 완료 ===")
print("\n사용 가능한 함수:")
print("  - analyze_gt_results(): GT 결과 품질 분석")
print("  - create_final_dataset(): 최종 데이터셋 생성 및 저장")
print("  - visualize_dataset_quality(): 품질 시각화")
print("\n실행 예시:")
print("# 1. GT 결과 분석")
print("analysis = analyze_gt_results(gt_results)")
print("# 2. 시각화")
print("visualize_dataset_quality(analysis, save_path=output_path)")
print("# 3. 최종 데이터셋 생성")
print("final_dataset = create_final_dataset(gt_results, output_path, min_confidence=0.6)")

In [ ]:
# 1. GT 결과 분석
analysis = analyze_gt_results(gt_results)
# 2. 시각화
visualize_dataset_quality(analysis, save_path=output_path)
# 3. 최종 데이터셋 생성
final_dataset = create_final_dataset(gt_results, output_path, min_confidence=0.6)